# Аналитика закупок и запасов (демо-версия ноутбука)

Ноутбук показывает этапы обработки данных по контрактам, потребности и остаткам
(склад фармацевтической компании). Все данные обезличены: производители, МНН,
наименования и суммы заменены псевдонимами и условными единицами.
Структура и логика анализа совпадает с боевой версией.

In [1]:
#pip install autopep8

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta
import math
import os
import glob

pd.set_option('display.max_columns', None)

In [3]:
#pip install XlsxWriter

In [4]:
df_columns_ispoln = ['Производитель',
                     'МНН',
                     'Наименование полное',
                     'Код, Номенклатура',
                     'Количество упаковок по ГК',
                     'Количество осталось отгрузить в целом',
                     'Цена закупки с НДС (согласованная)',
                     'Цена ГК за упак. с НДС (нов.)',
                     'Договор.Дата окончания действия',
                     'Количество упаковок в заявке',
                     'Код строки',
                     'Заказчик в шапке',
                     'Комментарий',
                    'Номер аукциона',
                    'Отгрузка по заявке',
                    'График поставки дата окончания периода']

In [5]:
company_mapping = {
    # Словарь с новыми названиями производителей
    'Производитель, д.д.': 'Производитель',
    'Производитель САС':'Производитель С.а.С.',
}

#### Загрузка данных

In [6]:
# Подгрузка данных
df_ispoln = pd.read_excel('//fs/SCAN/user078/Исполнение по строкам.xlsx', header = 3)

df_ostatki_1p = pd.read_excel('//fs/SCAN/user078/Остатки и доступность товаров.xlsx')
df_ostatki_2p = pd.read_excel('//fs/SCAN/user078/Остатки и доступность товаров.xlsx', header=1)
ostatki = df_ostatki_1p[1:].reset_index(drop=True).join(df_ostatki_2p)
df_ostatki = ostatki.loc[:, ~ostatki.columns.str.contains('Unnamed')]
df_ostatki = df_ostatki.drop(columns='Сейчас', axis=1)[:-1]

#df_ostatki = df_ostatki[:-1]

df_defect = pd.read_excel('//fs/SCAN/user078/Дефектура.xlsx')

price = pd.read_excel('//fs/SCAN/user078/Закупочные цены.xlsx')
price = price[:-1]
price = price.drop_duplicates().reset_index(drop=True)

#### Остатки и доступность

In [2]:
# Пример данных, с которыми работал
demo_overstock = pd.DataFrame([
    {
        "Номенклатура_анон": "Препарат_001",
        "Производитель_анон": "Производитель_001",
        "Категория": "Субстанция",
        "Серия.Годен до": "10.12.2025",
        "Срок годности": None,
        "В наличии_анон": 7.5,
        "В резерве_анон": 0.0,
        "Доступно_анон": 7.5,
        "Отгружается_анон": 0.0,
    },
    {
        "Номенклатура_анон": "Препарат_002",
        "Производитель_анон": "Производитель_002",
        "Категория": "ГЛС СТОК",
        "Серия.Годен до": "30.04.2028",
        "Срок годности": 36.0,
        "В наличии_анон": 5.3,
        "В резерве_анон": 0.0,
        "Доступно_анон": 5.3,
        "Отгружается_анон": 0.0,
    },
    {
        "Номенклатура_анон": "Препарат_002",
        "Производитель_анон": "Производитель_002",
        "Категория": "ГЛС СТОК",
        "Серия.Годен до": "30.04.2028",
        "Срок годности": 36.0,
        "В наличии_анон": 0.1,
        "В резерве_анон": 0.0,
        "Доступно_анон": 0.1,
        "Отгружается_анон": 0.0,
    },
])

demo_overstock

,Номенклатура_анон,Производитель_анон,Категория,Серия.Годен до,Срок годности,В наличии_анон,В резерве_анон,Доступно_анон,Отгружается_анон
0,Препарат_001,Производитель_001,Субстанция,10.12.2025,NaN,7.5,0.0,7.5,0.0
1,Препарат_002,Производитель_002,ГЛС СТОК,30.04.2028,36.0,5.3,0.0,5.3,0.0
2,Препарат_002,Производитель_002,ГЛС СТОК,30.04.2028,36.0,0.1,0.0,0.1,0.0


In [8]:
df_ostatki['chain'] = (df_ostatki['Номенклатура.Код'].astype(str) + ' ' +
                    df_ostatki['Характеристика.Документ поступления'] + ' ' +
                    df_ostatki['Номенклатура'] + ' ' +
                    df_ostatki['Характеристика'])

In [9]:
price['chain'] = (price['Код'].astype(str) + ' ' +
                  price['Документ поступления'] + ' ' +
                  price['Номенклатура'] + ' ' +
                  price['Характеристика'])

In [10]:
price = price.drop_duplicates(subset='chain')

In [11]:
df_ostatki_price = pd.merge(df_ostatki, price, on='chain', how='left')

In [12]:
columns_fill = ['Характеристика.Документ поступления', 'Характеристика.Серия.Годен до', 'Номенклатура.МНН']
df_ostatki_price[columns_fill] = df_ostatki_price[columns_fill].fillna('~')
df_ostatki_price['Номенклатура.Срок годности'] = df_ostatki_price['Номенклатура.Срок годности'].fillna(0.)

In [13]:
df_ostatki_price.info(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7810 entries, 0 to 7809
Data columns (total 19 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Номенклатура.Код                     7810 non-null   object 
 1   Номенклатура.МНН                     7810 non-null   object 
 2   Номенклатура.Товарная категория      7810 non-null   object 
 3   Характеристика.Серия.Годен до        7810 non-null   object 
 4   Номенклатура.Срок годности           7810 non-null   float64
 5   Характеристика.Документ поступления  7810 non-null   object 
 6   Номенклатура.Производитель           7810 non-null   object 
 7   Номенклатура_x                       7810 non-null   object 
 8   Характеристика_x                     7810 non-null   object 
 9   В наличии                            7135 non-null   float64
 10  В резерве                            757 non-null    float64
 11  Доступно                      

In [14]:
df_ostatki_price = df_ostatki_price.groupby(['Номенклатура.Код', 'Номенклатура.МНН', 'Номенклатура.Товарная категория',
                                             'Номенклатура_x', 'Характеристика_x', 'Характеристика.Документ поступления',
                                             'Характеристика.Серия.Годен до', 'Номенклатура.Срок годности',
                                             'Номенклатура.Производитель']).agg({'В наличии':'sum',
                                                                                 'В резерве':'sum',
                                                                                 'Доступно':'sum',
                                                                                 'Отгружается':'sum',
                                                                                 'Цена':'mean'}).reset_index()

df_ostatki_price = df_ostatki_price.rename(columns={'Номенклатура_x':'Номенклатура', 'Характеристика_x':'Характеристика'})

In [16]:
df_ostatki_price = df_ostatki_price[(df_ostatki_price['В наличии'].notna()) & (df_ostatki_price['В наличии']>0)]

In [ ]:
mask = df_ostatki_price['Цена'].notna()
df_valid = df_ostatki_price[mask]
avg_by_code = (df_valid.groupby('Номенклатура.Код')
                 .apply(lambda g: (g['В наличии'] * g['Цена']).sum() / g['В наличии'].sum())
                 .rename('avg_price'))

# Общая средневзвешенная
overall_avg = (df_valid['В наличии'] * df_valid['Цена']).sum() / df_valid['В наличии'].sum()

# Заполняем пропуски
df_ostatki_price = df_ostatki_price.merge(avg_by_code, on='Номенклатура.Код', how='left')
df_ostatki_price['Цена'] = df_ostatki_price['Цена'].fillna(df_ostatki_price['avg_price']).fillna(overall_avg)
df_ostatki_price = df_ostatki_price.drop('avg_price', axis=1)

In [18]:
df_ostatki_price['В наличии'].sum()

np.float64(2850879.6720000003)

In [19]:
# Преобразуем в datetime, затем форматируем как строку даты
df_ostatki_price['Годен до'] = pd.to_datetime(
    df_ostatki_price['Характеристика.Серия.Годен до'], 
    format='%d.%m.%Y %H:%M:%S'  # Явно указываем формат
).dt.strftime('%d.%m.%Y')
df_ostatki_price = df_ostatki_price.drop('Характеристика.Серия.Годен до', axis=1)

In [ ]:
df_ostatki_price.head(3)

In [21]:
df_ostatki_price['В наличии'] = df_ostatki_price['В наличии'] - df_ostatki_price['Отгружается']

In [22]:
df_ostatki_price = df_ostatki_price.rename(columns={'Характеристика.Документ поступления':'Документ поступления',
                                                   'Номенклатура':'Наименование полное',
                                                   'В наличии':'Остатки',
                                                   'Номенклатура.Производитель':'Производитель'}).drop(columns=['Доступно', 'Отгружается'], axis=1)

In [24]:
df_ostatki_price['Цена'] = df_ostatki_price['Цена']/1.1 # Цена без НДС
df_ostatki_price['Себестоимость'] = df_ostatki_price['Остатки'] * df_ostatki_price['Цена']

In [ ]:
df_ostatki_price['Себестоимость'].sum()

In [ ]:
df_ostatki_price.head(3)

#### Без количеств

In [27]:
without_counts = '//fs/SCAN/user078/Без количеств_2.xlsx'

In [28]:
skip_rows = 3
# Проверим так ли это
# Открываем файл и читаем строки до тех пор, пока не найдем заголовок
with pd.ExcelFile(without_counts) as xls:
    # Предполагаем, что шапка не превышает 100 строк
    for i in range(100):
        df_temp = pd.read_excel(xls,
                                nrows=1,
                                skiprows=i)
        # Ищем название первого столбца
        if df_temp.columns[0] == 'Код договора':
            print("break")
            skip_rows = i
            break
print("skip_rows _", skip_rows)

break
skip_rows _ 3


In [ ]:
without_counts = pd.read_excel(
    '//fs/SCAN/user078/Без количеств_2.xlsx',
    skiprows=skip_rows,
    skipfooter=1,
    converters= {'Ссылка.Номер извещения': str})
without_counts.head(2)

In [ ]:
# Получаем индексы столбцов с пропущенными значениями
unnamed_columns = [i for i, col in enumerate(without_counts.columns) if col.startswith('Unnamed')]
# Удаляем столбцы по индексам
without_counts = without_counts.drop(without_counts.columns[unnamed_columns], axis=1)
without_counts = without_counts.rename(columns={'Ссылка.Номер извещения':'Номер аукциона',
                                                'Наименование полное':'Наименование полное',
                                                'Производитель':'Производитель',
                                                'Цена ГКЗа упаковку':'Цена ГК за упак. с НДС (нов.)',
                                                'Среднее количество по строке':'Без количеств',
                                                'Ссылка.Заказчик':'Заказчик в шапке',
                                                'Ссылка.Заказчик.ИНН':'Заказчик в шапке.ИНН'})
without_counts.head(2)

In [3]:
# Пример данных
demo_contracts = pd.DataFrame([
    {
        "Код договора_анон": "Договор_001",
        "Код строки": 1,
        "Количество строк": 1,
        "Номер аукциона_анон": "Аукцион_001",
        "Источник тип": "Подписание ГК",
        "Заказчик_анон": "Заказчик_001",
        "МНН_анон": "МНН_001",
        "Наименование полное_анон": "Препарат_полный_001",
        "Номенклатура_анон": "Препарат_001",
        "Производитель_анон": "Производитель_001",
        "Региональная наценка": 17.0,
        "Рег. цена без НДС_анон": 14.3,
        "Средняя сумма по строке_анон": 10.4,
        "Цена ГК с НДС_анон": 12.8,
        "Цена закупки ГК_анон": 8.8,
        "Дата окончания контракта": "30.12.2025",
        "Код номенклатуры_анон": "Ном_001",
        "Макс. кол-во": 1,
        "Без количеств": 1,
        "Сумма ГК_анон": 10.4,
        "Остаток_анон": 0.1,
    },
    {
        "Код договора_анон": "Договор_002",
        "Код строки": 1,
        "Количество строк": 1,
        "Номер аукциона_анон": "Аукцион_002",
        "Источник тип": "Предложение КП",
        "Заказчик_анон": "Заказчик_002",
        "МНН_анон": "МНН_002",
        "Наименование полное_анон": "Препарат_полный_002",
        "Номенклатура_анон": "Препарат_002",
        "Производитель_анон": "Производитель_002",
        "Региональная наценка": 14.0,
        "Рег. цена без НДС_анон": 28.3,
        "Средняя сумма по строке_анон": 197.5,
        "Цена ГК с НДС_анон": 28.9,
        "Цена закупки ГК_анон": 25.4,
        "Дата окончания контракта": None,
        "Код номенклатуры_анон": "Ном_002",
        "Макс. кол-во": 41,
        "Без количеств": 7,
        "Сумма ГК_анон": 1184.7,
        "Остаток_анон": 0.0,
    }
])

demo_contracts

,Код договора_анон,Код строки,Количество строк,Номер аукциона_анон,Источник тип,Заказчик_анон,МНН_анон,Наименование полное_анон,Номенклатура_анон,Производитель_анон,Региональная наценка,Рег. цена без НДС_анон,Средняя сумма по строке_анон,Цена ГК с НДС_анон,Цена закупки ГК_анон,Дата окончания контракта,Код номенклатуры_анон,Макс. кол-во,Без количеств,Сумма ГК_анон,Остаток_анон
0,Договор_001,1,1,Аукцион_001,Подписание ГК,Заказчик_001,МНН_001,Препарат_полный_001,Препарат_001,Производитель_001,17.0,14.3,10.4,12.8,8.8,30.12.2025,Ном_001,1,1,10.4,0.1
1,Договор_002,1,1,Аукцион_002,Предложение КП,Заказчик_002,МНН_002,Препарат_полный_002,Препарат_002,Производитель_002,14.0,28.3,197.5,28.9,25.4,NaN,Ном_002,41,7,1184.7,0.0


In [ ]:
# Ищем УТ, которые отсутствуют в Итоге
uts = [
    "УТ-00024126",
    "УТ-00023140",
    "УТ-00007423",
    "УТ-00022852",
    "УТ-00022101",
    ]
df = (
    without_counts.loc[without_counts["Номенклатура.Код"].isin(uts), :]
    .drop_duplicates(subset=[
        "Производитель",
        "Наименование полное"
        ]
        )
    )
df

In [32]:
iskl = pd.read_excel('//fs/SCAN/user078/Исключения для без количеств.xlsx',
                      converters={'Номер аукциона': str})

In [33]:
without_counts = without_counts.merge(iskl,
                                      on=['Номер аукциона', 'Наименование полное', 'Производитель'],
                                      how='left')

In [34]:
without_counts.loc[without_counts['Производитель правильный'].isna() == False,
                    'Производитель'
                    ] = without_counts.loc[without_counts['Производитель правильный'].isna() == False,
                                                           'Производитель правильный']

In [35]:
without_counts = without_counts.drop('Производитель правильный', axis=1)

In [36]:
without_counts = without_counts[['Номер аукциона','Номенклатура.Код',
                                 'Заказчик в шапке','Заказчик в шапке.ИНН',
                                 'Наименование полное','Производитель',
                                 'Цена ГК за упак. с НДС (нов.)','Без количеств']]

In [37]:
without_counts['chain'] = (without_counts['Номер аукциона'] + ' ' +
                           without_counts['Заказчик в шапке'] + ' ' +
                           without_counts['Наименование полное'] + ' ' +
                           without_counts['Производитель'] + ' ' +
                           without_counts['Цена ГК за упак. с НДС (нов.)'].astype(str)
                           )

In [38]:
without_counts.Производитель

0           Эллара ООО
1       Брынцалов-А АО
2        Велфарм-М ООО
3       Брынцалов-А АО
4          Мосфарм ООО
             ...      
1066    Брынцалов-А АО
1067       Мосфарм ООО
1068       Мосфарм ООО
1069        Алтегра АО
1070        Алтегра АО
Name: Производитель, Length: 1071, dtype: object

In [39]:
without_counts.shape

(1071, 9)

In [40]:
without_counts = without_counts.drop_duplicates(subset='chain')

In [ ]:
without_counts.head(2)

#### Расчет остатков и в резерве

In [42]:
df_ostatki = df_ostatki_price.copy()

In [43]:
df_ostatki.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6921 entries, 0 to 6920
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Номенклатура.Код                 6921 non-null   object 
 1   Номенклатура.МНН                 6921 non-null   object 
 2   Номенклатура.Товарная категория  6921 non-null   object 
 3   Наименование полное              6921 non-null   object 
 4   Характеристика                   6921 non-null   object 
 5   Документ поступления             6921 non-null   object 
 6   Номенклатура.Срок годности       6921 non-null   float64
 7   Производитель                    6921 non-null   object 
 8   Остатки                          6921 non-null   float64
 9   В резерве                        6921 non-null   float64
 10  Цена                             6921 non-null   float64
 11  Годен до                         6921 non-null   object 
 12  Себестоимость       

In [44]:
df_ostatki['Наименование полное'] = df_ostatki['Наименование полное'].str.extract(r'^(.*?\))(?:\s|$)|^([^-]+)(?:\s-|$)').fillna('').sum(axis=1)

# Если результат пустой, берем оригинальный текст
mask = df_ostatki['Наименование полное'] == ''
df_ostatki.loc[mask, 'Наименование полное'] = df_ostatki.loc[mask, 'Наименование полное']

In [ ]:
df_ostatki.head(2)

In [46]:
def clean_manufacturer(manufacturer):
    if pd.isna(manufacturer):
        return manufacturer
    
    manufacturer = str(manufacturer)
    
    # Define the countries list within the function scope
    countries = ['россия', 'сша', 'китай', 'германия', 'франция', 'италия', 'великобритания',
                'индия', 'испания', 'мальта', 'кипр', 'словакия', 'иран', 'турция', 'япония',
                'чехия', 'ирландия', 'дания', 'португалия', 'босния', 'герцеговина', 'черногория',
                'македония', 'румыния', 'бельгия', 'польша', 'болгария', 'беларусь', 'венгрия', 'словения',
                'австрия', 'армения', 'канада', 'хорватия', 'швейцария', 'соединенное королевство', 'украина',
                'сербия', 'швеция', 'греция', 'нидерланды', 'чешская республика']
    
    # Случай 1: Есть запятая с указанием страны
    if ',' in manufacturer:
        parts = manufacturer.split(',')
        last_part = parts[-1].lower()
        
        if any(country in last_part for country in countries):
            return ', '.join(parts[:-1]).strip()
    
    # Случай 2: Есть дефис с указанием страны
    if ' - ' in manufacturer:
        parts = manufacturer.split(' - ')
        last_part = parts[-1].lower()
        if any(country in last_part for country in countries):
            return ' - '.join(parts[:-1]).strip()
    
    # Случай 3: Есть косая черта с указанием местного представителя
    if '/' in manufacturer:
        parts = manufacturer.split('/')
        # Оставляем только первую часть (основного производителя)
        return parts[0].strip()
    
    # Случай 4: Убираем "РОССИЯ" в конце без запятой
    if manufacturer.endswith('РОССИЯ'):
        manufacturer = manufacturer[:-6].strip()
    elif manufacturer.endswith('РОССИЯ,'):
        manufacturer = manufacturer[:-7].strip()
    
    return manufacturer.strip()

# Применяем функцию к столбцу
df_ostatki['Производитель'] = df_ostatki['Производитель'].apply(clean_manufacturer)

# Дополнительная очистка для специфических случаев
def additional_cleaning(manufacturer):
    if pd.isna(manufacturer):
        return manufacturer
    
    manufacturer = str(manufacturer)
    
    # Убираем лишние пробелы
    manufacturer = ' '.join(manufacturer.split())
    
    return manufacturer

# Применяем дополнительную очистку
df_ostatki['Производитель'] = df_ostatki['Производитель'].apply(additional_cleaning)

In [47]:
df_ostatki['Производитель'] = df_ostatki['Производитель'].replace(company_mapping)

In [48]:
df_ostatki = df_ostatki.fillna('~')

In [49]:
df_ostatki.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6921 entries, 0 to 6920
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Номенклатура.Код                 6921 non-null   object 
 1   Номенклатура.МНН                 6921 non-null   object 
 2   Номенклатура.Товарная категория  6921 non-null   object 
 3   Наименование полное              6921 non-null   object 
 4   Характеристика                   6921 non-null   object 
 5   Документ поступления             6921 non-null   object 
 6   Номенклатура.Срок годности       6921 non-null   float64
 7   Производитель                    6921 non-null   object 
 8   Остатки                          6921 non-null   float64
 9   В резерве                        6921 non-null   float64
 10  Цена                             6921 non-null   float64
 11  Годен до                         6921 non-null   object 
 12  Себестоимость       

In [50]:
# df_ostatki_gr = df_ostatki.groupby(['Номенклатура.Код', 'Производитель', 'Номенклатура.МНН',
#                                  'Документ поступления', 'Наименование полное',
#                                  'Характеристика', 'Годен до']).agg({'Остатки':'sum',
#                                                                      'В резерве':'sum'}).reset_index()

#### Добавляем фактическую цену закупки

In [51]:
merged_df = df_ostatki.copy()

In [52]:
merged_df = merged_df.rename(columns={'Цена':'Цена за единицу', 'Номенклатура.Срок годности':'Срок годности'})
merged_df['Остатки, руб.'] = merged_df['Себестоимость']
merged_df['В резерве, руб.'] = merged_df['Цена за единицу'] * merged_df['В резерве']

In [53]:
merged_df = merged_df.rename(columns={'Себестоимость':'Цена'})
# Преобразуем в datetime
merged_df['Годен до'] = pd.to_datetime(merged_df['Годен до'], format='%d.%m.%Y', errors='coerce')
merged_df = merged_df.rename(columns = {'Остатки' : 'Остатки, уп.', 'В резерве' : 'В резерве, уп.'})

In [54]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6921 entries, 0 to 6920
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Номенклатура.Код                 6921 non-null   object        
 1   Номенклатура.МНН                 6921 non-null   object        
 2   Номенклатура.Товарная категория  6921 non-null   object        
 3   Наименование полное              6921 non-null   object        
 4   Характеристика                   6921 non-null   object        
 5   Документ поступления             6921 non-null   object        
 6   Срок годности                    6921 non-null   float64       
 7   Производитель                    6921 non-null   object        
 8   Остатки, уп.                     6921 non-null   float64       
 9   В резерве, уп.                   6921 non-null   float64       
 10  Цена за единицу                  6921 non-null   float64    

In [55]:
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Текущая дата
current_date = datetime.now()

def calculate_metrics(row):
    # Преобразуем дату в datetime
    expiry_date = row['Годен до']#datetime.strptime(row['Годен до'], '%d.%m.%Y')
    total_months = row['Срок годности']
    
    # Рассчитываем оставшийся срок в месяцах (дробно, если меньше 1 месяца)
    remaining_days = (expiry_date - current_date).days
    
    if remaining_days <= 0:
        remaining_months = 0
    elif remaining_days < 30:
        # Если меньше месяца, считаем дробно
        remaining_months = remaining_days / 30.44  # среднее количество дней в месяце
    else:
        # Если больше месяца, используем relativedelta для точного расчета
        delta = relativedelta(expiry_date, current_date)
        remaining_months = delta.years * 12 + delta.months + delta.days / 30.44
    
    # Рассчитываем процент от общего срока
    if total_months > 0:
        percentage = (remaining_months / total_months) * 100
    else:
        percentage = 0
    
    return round(remaining_months, 1), round(percentage)

# Применяем функцию к каждой строке
merged_df['Остаток СГ'] = merged_df.apply(lambda row: calculate_metrics(row)[0], axis=1)
merged_df['ОСГ, %'] = merged_df.apply(lambda row: calculate_metrics(row)[1], axis=1)

# Выводим результат
print(merged_df[['Годен до', 'Срок годности', 'Остаток СГ', 'ОСГ, %']])

       Годен до  Срок годности  Остаток СГ  ОСГ, %
0    2026-06-01            0.0         5.9       0
1    2028-09-01            0.0        32.9       0
2    2028-10-01            0.0        33.8       0
3    2029-06-01            0.0        41.9       0
4    2029-06-01            0.0        41.9       0
...         ...            ...         ...     ...
6916 2028-04-30           36.0        28.8      80
6917 2027-01-31           36.0        13.8      38
6918 2028-06-30           36.0        30.8      86
6919 2028-09-16            0.0        33.3       0
6920 2028-10-30           36.0        34.8      97

[6921 rows x 4 columns]


In [ ]:
merged_df.head(3)

In [5]:
demo = pd.DataFrame([
    {
        "Код_анон": "Ном_001",
        "МНН_анон": "МНН_A",
        "Категория": "Субстанция",
        "Наименование_анон": "Субстанция_препарат_001",
        "Документ поступления_анон": "Документ_001",
        "Срок годности": 0.0,
        "Производитель_анон": "Производитель_001",
        "Остатки_уп_анон": 5.0,         # 50.0 / 10
        "В резерве_уп_анон": 0.0,
        "Цена за единицу_анон": 293.58, # округлённое значение
        "Годен до": "2026-06-01",
        "Остатки_руб_анон": 1467.89,    # 14678.86 / 10
        "В резерве_руб_анон": 1467.89,
        "Остаток ОСГ_анон": 0.0,
        "ОСГ_анон_%": 5.9,
    },
    {
        "Код_анон": "Ном_002",
        "МНН_анон": "МНН_A",
        "Категория": "Субстанция",
        "Наименование_анон": "Субстанция_препарат_002",
        "Документ поступления_анон": "Документ_002",
        "Срок годности": 0.0,
        "Производитель_анон": "Производитель_002",
        "Остатки_уп_анон": 35.0,        # 350.0 / 10
        "В резерве_уп_анон": 0.0,
        "Цена за единицу_анон": 2358.83,
        "Годен до": "2028-09-01",
        "Остатки_руб_анон": 82558.95,   # 825589.55 / 10
        "В резерве_руб_анон": 82558.95,
        "Остаток ОСГ_анон": 0.0,
        "ОСГ_анон_%": 32.9,
    },
    {
        "Код_анон": "Ном_002",
        "МНН_анон": "МНН_A",
        "Категория": "Субстанция",
        "Наименование_анон": "Субстанция_препарат_002",
        "Документ поступления_анон": "Документ_003",
        "Срок годности": 0.0,
        "Производитель_анон": "Производитель_002",
        "Остатки_уп_анон": 30.0,        # 300.0 / 10
        "В резерве_уп_анон": 0.0,
        "Цена за единицу_анон": 2251.97,
        "Годен до": "2028-10-01",
        "Остатки_руб_анон": 67559.18,   # 675591.82 / 10
        "В резерве_руб_анон": 67559.18,
        "Остаток ОСГ_анон": 0.0,
        "ОСГ_анон_%": 33.8,
    },
])

demo

,Код_анон,МНН_анон,Категория,Наименование_анон,Документ поступления_анон,Срок годности,Производитель_анон,Остатки_уп_анон,В резерве_уп_анон,Цена за единицу_анон,Годен до,Остатки_руб_анон,В резерве_руб_анон,Остаток ОСГ_анон,ОСГ_анон_%
0,Ном_001,МНН_A,Субстанция,Субстанция_препарат_001,Документ_001,0.0,Производитель_001,5.0,0.0,293.58,2026-06-01,1467.89,1467.89,0.0,5.9
1,Ном_002,МНН_A,Субстанция,Субстанция_препарат_002,Документ_002,0.0,Производитель_002,35.0,0.0,2358.83,2028-09-01,82558.95,82558.95,0.0,32.9
2,Ном_002,МНН_A,Субстанция,Субстанция_препарат_002,Документ_003,0.0,Производитель_002,30.0,0.0,2251.97,2028-10-01,67559.18,67559.18,0.0,33.8


In [57]:
result_ostatki = merged_df.groupby(['Производитель', 'Номенклатура.Код', 'Наименование полное', 'Номенклатура.МНН',
                                   'Остаток СГ', 'ОСГ, %', 'Годен до', 'Цена за единицу']).agg({'Остатки, уп.' : 'sum','Остатки, руб.' : 'sum',
                                                                                                'В резерве, уп.' : 'sum', 'В резерве, руб.' : 'sum',
                                                                                                'Цена':'sum'}).reset_index()

In [ ]:
result_ostatki['Цена'].sum()

In [59]:
#result_ostatki = result_ostatki[result_ostatki['Остатки, уп.'] > 0]

In [ ]:
result_ostatki.head(3)

#### Дефектура

In [61]:
#df_defect = df_defect.dropna(subset='Вид дефектуры')

In [62]:
df_defect.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1085 entries, 0 to 1084
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Период                           1085 non-null   object
 1   Номенклатура                     1085 non-null   object
 2   Код                              1085 non-null   object
 3   Вид дефектуры                    874 non-null    object
 4   Дата снятия отметки о дефектуре  350 non-null    object
 5   Предполагаемая дата поставки     145 non-null    object
 6   Комментарий                      536 non-null    object
 7   Ответственный                    522 non-null    object
dtypes: object(8)
memory usage: 67.9+ KB


In [ ]:
df_defect[df_defect['Код']=='УТ-00010184']

In [64]:
df_defect['Период'] = pd.to_datetime(df_defect['Период'], format='%d.%m.%Y %H:%M:%S')
#df_defect = df_defect.sort_values('Период').groupby('Код').agg({'Вид дефектуры':'last', 'Дата снятия отметки о дефектуре':'last'})

df_defect['Дата снятия отметки о дефектуре'] = pd.to_datetime(df_defect['Дата снятия отметки о дефектуре'], format='%d.%m.%Y %H:%M:%S')
df_defect['Дата снятия отметки о дефектуре'] = df_defect['Дата снятия отметки о дефектуре'].fillna(pd.to_datetime('01.01.2030 00:00:00',
                                                                                                                   format='%d.%m.%Y %H:%M:%S'))

# df_defect = df_defect.sort_values('Период').groupby(['Код', 'Вид дефектуры']).agg({'Дата снятия отметки о дефектуре':'last'}).reset_index()
#df_defect = df_defect.sort_values('Период').groupby('Код').last().reset_index()

# Находим индекс последней записи по дате для каждой группы
idx = df_defect.groupby('Код')['Период'].idxmax()

# Выбираем полные строки по найденным индексам
df_defect = df_defect.loc[idx].reset_index(drop=True)

In [65]:
df_defect = df_defect[['Код', 'Вид дефектуры', 'Дата снятия отметки о дефектуре', 'Комментарий']].dropna(subset='Вид дефектуры')

In [66]:
df_defect.info()

<class 'pandas.core.frame.DataFrame'>
Index: 544 entries, 0 to 681
Data columns (total 4 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Код                              544 non-null    object        
 1   Вид дефектуры                    544 non-null    object        
 2   Дата снятия отметки о дефектуре  544 non-null    datetime64[ns]
 3   Комментарий                      282 non-null    object        
dtypes: datetime64[ns](1), object(3)
memory usage: 21.2+ KB


In [ ]:
df_defect.head()

In [68]:
# Фильтруем по действующей дефектуре от сегодняшнего дня
df_defect = df_defect[df_defect['Дата снятия отметки о дефектуре'] > datetime.now()]

In [69]:
df_defect.info()

<class 'pandas.core.frame.DataFrame'>
Index: 443 entries, 0 to 681
Data columns (total 4 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Код                              443 non-null    object        
 1   Вид дефектуры                    443 non-null    object        
 2   Дата снятия отметки о дефектуре  443 non-null    datetime64[ns]
 3   Комментарий                      211 non-null    object        
dtypes: datetime64[ns](1), object(3)
memory usage: 17.3+ KB


#### Расчет "В заявке" и "Потребности"

In [71]:
df_ispoln.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18672 entries, 0 to 18671
Data columns (total 33 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Дата исполнения                                 18672 non-null  object 
 1   Unnamed: 1                                      0 non-null      float64
 2   Unnamed: 2                                      0 non-null      float64
 3   На исполнении                                   18671 non-null  object 
 4   Производитель                                   18671 non-null  object 
 5   МНН                                             18661 non-null  object 
 6   Unnamed: 6                                      0 non-null      float64
 7   Наименование полное                             18671 non-null  object 
 8   Номер аукциона                                  18671 non-null  object 
 9   Количество упаковок по ГК              

In [72]:
df_ispoln = df_ispoln.drop_duplicates()
df_ispoln = df_ispoln.drop(df_ispoln.index[-1])
df_ispoln = df_ispoln[df_columns_ispoln]
df_ispoln.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18155 entries, 0 to 18670
Data columns (total 16 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Производитель                           18155 non-null  object 
 1   МНН                                     18145 non-null  object 
 2   Наименование полное                     18155 non-null  object 
 3   Код, Номенклатура                       18155 non-null  object 
 4   Количество упаковок по ГК               18145 non-null  float64
 5   Количество осталось отгрузить в целом   11805 non-null  float64
 6   Цена закупки с НДС (согласованная)      17500 non-null  float64
 7   Цена ГК за упак. с НДС (нов.)           17756 non-null  float64
 8   Договор.Дата окончания действия         18155 non-null  object 
 9   Количество упаковок в заявке            1026 non-null   float64
 10  Код строки                              18155 non-null  float64

In [73]:
df_ispoln['В заявке'] = np.where(
    df_ispoln['Отгрузка по заявке'] == 'Да',
    df_ispoln['Количество упаковок в заявке'],
    None  # или 0, np.nan — в зависимости от требований
)

df_ispoln['По графику'] = np.where(
    df_ispoln['Отгрузка по заявке'] != 'Да',
    df_ispoln['Количество упаковок в заявке'],
    None  # или 0, np.nan
)

In [74]:
mask = df_ispoln['Код, Номенклатура'] == ', '
df_ispoln.loc[mask, 'Код, Номенклатура'] = ', ' + df_ispoln.loc[mask, 'Наименование полное']

In [75]:
df_ispoln.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18155 entries, 0 to 18670
Data columns (total 18 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Производитель                           18155 non-null  object 
 1   МНН                                     18145 non-null  object 
 2   Наименование полное                     18155 non-null  object 
 3   Код, Номенклатура                       18155 non-null  object 
 4   Количество упаковок по ГК               18145 non-null  float64
 5   Количество осталось отгрузить в целом   11805 non-null  float64
 6   Цена закупки с НДС (согласованная)      17500 non-null  float64
 7   Цена ГК за упак. с НДС (нов.)           17756 non-null  float64
 8   Договор.Дата окончания действия         18155 non-null  object 
 9   Количество упаковок в заявке            1026 non-null   float64
 10  Код строки                              18155 non-null  float64

In [76]:
# Простой способ - разделяем по запятой и берем вторую часть, затем обрезаем по производителю
df_ispoln['Номенклатура.Код'] = df_ispoln['Код, Номенклатура'].str.split(', ').str[0]
df_ispoln['Наименование полное'] = df_ispoln['Код, Номенклатура'].str.split(', ').str[1]

# Убираем производителя (все что после закрывающей скобки)
df_ispoln['Наименование полное'] = df_ispoln['Наименование полное'].str.split('\)').str[0] + ')'

# Обработка случая с ","
mask = df_ispoln['Код, Номенклатура'].str.strip() == ','
df_ispoln.loc[mask, ['Номенклатура.Код', 'Наименование полное']] = np.nan

In [77]:
# Маска: строки, где "Код, Номенклатура" начинается с ", "
mask = df_ispoln['Код, Номенклатура'].str.startswith(', ', na=False)

# Для таких строк берём всё после ", " (без пробела) и пишем в "Наименование полное"
df_ispoln.loc[mask, 'Наименование полное'] = df_ispoln['Код, Номенклатура'].str[2:]

In [ ]:
df_ispoln[['Код, Номенклатура', 'Номенклатура.Код', 'Наименование полное']].head(10)

In [ ]:
df_ispoln[['Номенклатура.Код', 'Производитель', 'Номер аукциона', 'Наименование полное']].head()

In [ ]:
without_counts[['Номенклатура.Код', 'Производитель', 'Номер аукциона', 'Наименование полное', 'Без количеств']].head()

In [81]:
df_ispoln['Количество осталось отгрузить в целом'].sum()

np.float64(5036218.829)

In [82]:
without_counts['Без количеств'].sum()

np.int64(879147)

In [83]:
df_ispoln = pd.merge(df_ispoln, without_counts[['Номенклатура.Код', 'Производитель', 'Номер аукциона', 'Заказчик в шапке', 'Без количеств']],
                     on=['Номенклатура.Код', 'Производитель', 'Заказчик в шапке', 'Номер аукциона'], how='left')

In [84]:
df_ispoln['Количество осталось отгрузить в целом'].sum()

np.float64(5036218.829)

In [85]:
df_ispoln['Без количеств'].sum()

np.float64(841186.0)

In [86]:
df_ispoln[['Номенклатура.Код', 'Производитель', 'Номер аукциона', 'Наименование полное', 'Без количеств']].to_excel('test.xlsx')

In [87]:
df_ispoln['МНН'] = df_ispoln['МНН'].fillna('~')

In [88]:
df_ispoln.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18157 entries, 0 to 18156
Data columns (total 20 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Производитель                           18157 non-null  object 
 1   МНН                                     18157 non-null  object 
 2   Наименование полное                     18157 non-null  object 
 3   Код, Номенклатура                       18157 non-null  object 
 4   Количество упаковок по ГК               18146 non-null  float64
 5   Количество осталось отгрузить в целом   11805 non-null  float64
 6   Цена закупки с НДС (согласованная)      17502 non-null  float64
 7   Цена ГК за упак. с НДС (нов.)           17758 non-null  float64
 8   Договор.Дата окончания действия         18157 non-null  object 
 9   Количество упаковок в заявке            1026 non-null   float64
 10  Код строки                              18157 non-null  fl

In [89]:
df_ispoln['Количество осталось отгрузить в целом'] = df_ispoln['Количество осталось отгрузить в целом'].fillna(0) + df_ispoln['Без количеств'].fillna(0)

In [90]:
df_ispoln['Цена ГК за упак. с НДС (нов.)'] = df_ispoln['Цена ГК за упак. с НДС (нов.)'].fillna(0)

In [91]:
df_ispoln = df_ispoln.rename(columns={'Комментарий':'Комментарий расторжение'})

In [92]:
df_ispoln['Комментарий расторжение'] = df_ispoln['Комментарий расторжение'].fillna('~')

In [93]:
isp_gr = df_ispoln.groupby(['Номенклатура.Код', 'Производитель',
                           'МНН', 'Наименование полное', 'Договор.Дата окончания действия', 'График поставки дата окончания периода',
                          'Цена ГК за упак. с НДС (нов.)']).agg({'Комментарий расторжение':'first',
                                                                 'Количество осталось отгрузить в целом': 'sum',
                                                                 'В заявке':'sum', 'По графику':'sum'}).reset_index()
isp_gr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15194 entries, 0 to 15193
Data columns (total 11 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Номенклатура.Код                        15194 non-null  object 
 1   Производитель                           15194 non-null  object 
 2   МНН                                     15194 non-null  object 
 3   Наименование полное                     15194 non-null  object 
 4   Договор.Дата окончания действия         15194 non-null  object 
 5   График поставки дата окончания периода  15194 non-null  object 
 6   Цена ГК за упак. с НДС (нов.)           15194 non-null  float64
 7   Комментарий расторжение                 15194 non-null  object 
 8   Количество осталось отгрузить в целом   15194 non-null  float64
 9   В заявке                                15194 non-null  object 
 10  По графику                              15194 non-null  ob

In [94]:
isp_gr[isp_gr['Номенклатура.Код']=='УТ-00025124']['Количество осталось отгрузить в целом'].sum()

np.float64(563805.0)

In [95]:
isp_gr['Цена ГК за упак. с НДС (нов.)'] = isp_gr['Цена ГК за упак. с НДС (нов.)'] / 1.1

In [ ]:
isp_gr.head()

In [97]:
isp_gr['Потребность, уп.'] = isp_gr['Количество осталось отгрузить в целом']
isp_gr['Потребность, руб.'] = round(isp_gr['Цена ГК за упак. с НДС (нов.)'] * isp_gr['Количество осталось отгрузить в целом'], 2)

isp_gr['В заявке, уп.'] = isp_gr['В заявке']
isp_gr['В заявке, руб.'] = round(isp_gr['В заявке'] * isp_gr['Цена ГК за упак. с НДС (нов.)'],2)

isp_gr['По графику, уп.'] = isp_gr['По графику']
isp_gr['По графику, руб.'] = round(isp_gr['По графику'] * isp_gr['Цена ГК за упак. с НДС (нов.)'],2)

In [98]:
isp_gr['Потребность, руб.'].sum()

np.float64(1727760560.5900002)

In [99]:
current_date = datetime.now()
isp_gr['Договор.Дата окончания действия'] = pd.to_datetime(isp_gr['Договор.Дата окончания действия'], format='%d.%m.%Y')
isp_gr['График поставки дата окончания периода'] = pd.to_datetime(isp_gr['График поставки дата окончания периода'], format='%d.%m.%Y')
isp_gr['Мес. до конца дог.'] =  ((isp_gr['Договор.Дата окончания действия'] - current_date).dt.days / 30.44).apply(math.ceil)

In [100]:
isp_gr['Год поставки'] = pd.to_datetime(isp_gr['График поставки дата окончания периода'], format='%d.%m.%Y').dt.strftime('%Y')

In [101]:
isp_gr['Потребность уп./мес.'] = round(isp_gr['Потребность, уп.'] /  isp_gr['Мес. до конца дог.'])
isp_gr['Потребность руб./мес.'] = round(isp_gr['Потребность, руб.'] / isp_gr['Мес. до конца дог.'])

In [ ]:
isp_gr.head()

In [6]:
# Пример данных
demo_demand = pd.DataFrame([
    {
        "Производитель_анон": "Производитель_001",
        "МНН_анон": "МНН_B",
        "Наименование_анон": "Препарат_закупка_001",
        "Дата окончания договора": "2026-12-31",
        "Дата окончания периода": "2026-03-31",
        "Цена ГК с НДС_анон": 95.45,          # условная цена за упаковку
        "Комментарий_анон": "Комментарий_001",
        "Кол-во осталось отгрузить_анон": 3.0,   # 30.000 / 10
        "В заявке_анон": 0.0,
        "По графику_анон": 0.0,
        "Потребность_уп_анон": 3.0,
        "Потребность_руб_анон": 2863.58,     # 28635.82 / 10
        "В заявке_уп_анон": 0.0,
        "В заявке_руб_анон": 0.0,
        "По графику_уп_анон": 0.0,
        "По графику_руб_анон": 0.0,
        "Мес. до конца дог.": 13,
        "Год поставки": 2026,
        "Потребность_уп/мес_анон": 2.0,
        "Потребность_руб/мес_анон": 220.3,
    },
    {
        "Производитель_анон": "Производитель_002",
        "МНН_анон": "МНН_C",
        "Наименование_анон": "Препарат_закупка_002",
        "Дата окончания договора": "2026-06-10",
        "Дата окончания периода": "2026-02-16",
        "Цена ГК с НДС_анон": 0.0,            # в исходных данных 0.0
        "Комментарий_анон": "Комментарий_002",
        "Кол-во осталось отгрузить_анон": 0.03,  # 0.333 / 10
        "В заявке_анон": 0.0,
        "По графику_анон": 0.0,
        "Потребность_уп_анон": 0.03,
        "Потребность_руб_анон": 0.0,
        "В заявке_уп_анон": 0.0,
        "В заявке_руб_анон": 0.0,
        "По графику_уп_анон": 0.0,
        "По графику_руб_анон": 0.0,
        "Мес. до конца дог.": 7,
        "Год поставки": 2026,
        "Потребность_уп/мес_анон": 0.0,
        "Потребность_руб/мес_анон": 0.0,
    },
    {
        "Производитель_анон": "Производитель_002",
        "МНН_анон": "МНН_C",
        "Наименование_анон": "Препарат_закупка_002",
        "Дата окончания договора": "2026-06-10",
        "Дата окончания периода": "2026-02-16",
        "Цена ГК с НДС_анон": 38.11,          # 381.127273 → условно 38.11
        "Комментарий_анон": "Комментарий_002",
        "Кол-во осталось отгрузить_анон": 6.5,   # 65.000 / 10
        "В заявке_анон": 0.0,
        "По графику_анон": 0.0,
        "Потребность_уп_анон": 6.5,
        "Потребность_руб_анон": 2477.33,     # 24773.27 / 10
        "В заявке_уп_анон": 0.0,
        "В заявке_руб_анон": 0.0,
        "По графику_уп_анон": 0.0,
        "По графику_руб_анон": 0.0,
        "Мес. до конца дог.": 7,
        "Год поставки": 2026,
        "Потребность_уп/мес_анон": 0.9,
        "Потребность_руб/мес_анон": 353.9,
    },
])

demo_demand

,Производитель_анон,МНН_анон,Наименование_анон,Дата окончания договора,Дата окончания периода,Цена ГК с НДС_анон,Комментарий_анон,Кол-во осталось отгрузить_анон,В заявке_анон,По графику_анон,Потребность_уп_анон,Потребность_руб_анон,В заявке_уп_анон,В заявке_руб_анон,По графику_уп_анон,По графику_руб_анон,Мес. до конца дог.,Год поставки,Потребность_уп/мес_анон,Потребность_руб/мес_анон
0,Производитель_001,МНН_B,Препарат_закупка_001,2026-12-31,2026-03-31,95.45,Комментарий_001,3.00,0.0,0.0,3.00,2863.58,0.0,0.0,0.0,0.0,13,2026,2.0,220.3
1,Производитель_002,МНН_C,Препарат_закупка_002,2026-06-10,2026-02-16,0.00,Комментарий_002,0.03,0.0,0.0,0.03,0.00,0.0,0.0,0.0,0.0,7,2026,0.0,0.0
2,Производитель_002,МНН_C,Препарат_закупка_002,2026-06-10,2026-02-16,38.11,Комментарий_002,6.50,0.0,0.0,6.50,2477.33,0.0,0.0,0.0,0.0,7,2026,0.9,353.9


In [103]:
isp_gr['МНН'] = isp_gr['МНН'].replace('Вода', 'Вода для инъекций')

In [104]:
# Проверяем дубликаты кодов перед группировкой
duplicate_codes = isp_gr['Номенклатура.Код'].duplicated().sum()
print(f"Дубликатов кодов перед группировкой: {duplicate_codes}")

# Группируем только по коду и дате окончания, остальные поля агрегируем первыми значениями
isp_gr = isp_gr.groupby(['Номенклатура.Код', 'Договор.Дата окончания действия', 'Год поставки']).agg({
    'МНН': 'first',
    'Наименование полное': 'first',
    'Комментарий расторжение':'first',
    'В заявке, уп.': 'sum',
    'В заявке, руб.': 'sum',
    'По графику, уп.': 'sum',
    'По графику, руб.': 'sum',
    'Потребность, уп.': 'sum',
    'Потребность, руб.': 'sum',
    'Потребность уп./мес.': 'sum',
    'Потребность руб./мес.': 'sum'
}).reset_index()

Дубликатов кодов перед группировкой: 13637


In [105]:
isp_gr['Потребность, руб.'].sum()

np.float64(1727760560.5900002)

In [106]:
# isp_gr = isp_gr.groupby(['Номенклатура.Код','МНН', 'Наименование полное', 'Договор.Дата окончания действия']).agg({'В заявке, уп.': 'sum', 'В заявке, руб.':'sum',
#                                                                                 'Потребность, уп.' : 'sum',
#                                                                                 'Потребность, руб.' : 'sum',
#                                                                                 'Потребность уп./мес.' : 'sum',
#                                                                                 'Потребность руб./мес.' : 'sum'}).reset_index()

In [107]:
isp_gr = isp_gr[~((isp_gr['В заявке, уп.']==0) & (isp_gr['По графику, уп.']==0) & (isp_gr['Потребность, уп.']==0) & (isp_gr['Потребность, руб.']==0) &
                 (isp_gr['Потребность уп./мес.']==0) & (isp_gr['Потребность руб./мес.']==0))]

In [108]:
isp_gr['МНН'] = isp_gr['МНН'].str.replace('\*', '', regex=True)

In [109]:
result_ostatki = result_ostatki.rename(columns={'Номенклатура.МНН':'МНН'})

In [110]:
isp_gr = pd.merge(
    isp_gr, 
    df_ispoln[['Номенклатура.Код', 'Производитель']].drop_duplicates('Номенклатура.Код', keep='first'), 
    on='Номенклатура.Код', 
    how='inner'
)

In [111]:
import re
# очистка ОПФ в наименовании производителя
def clean_company_name(company_name):
    if pd.isna(company_name):
        return company_name
    
    # Паттерн для поиска организационно-правовых форм
    pattern = r'\s*(?:ОАО|ПАО|АО|ЗАО|ООО|AB)\b\s*'
    
    # Удаляем формы и лишние пробелы
    cleaned = re.sub(pattern, ' ', str(company_name))
    cleaned = re.sub(r'\s+', ' ', cleaned)  # Убираем множественные пробелы
    
    return cleaned.strip()

isp_gr['Производитель'] = isp_gr['Производитель'].apply(clean_company_name)
result_ostatki['Производитель'] = result_ostatki['Производитель'].apply(clean_company_name)

In [112]:
isp_gr['Производитель'] = isp_gr['Производитель'].replace(company_mapping)

In [113]:
isp_gr[isp_gr['Номенклатура.Код'].isna()]

,Номенклатура.Код,Договор.Дата окончания действия,Год поставки,МНН,Наименование полное,Комментарий расторжение,"В заявке, уп.","В заявке, руб.","По графику, уп.","По графику, руб.","Потребность, уп.","Потребность, руб.",Потребность уп./мес.,Потребность руб./мес.,Производитель


In [114]:
# Заполняем отсутствующие УТ
ut_dict = {'Препарат_1':'Код-001',
          'Препарат_2':'Код-002'}

In [115]:
# Если нужно заменять не все значения, а только те, которые есть в словаре
mask = isp_gr['Наименование полное'].isin(ut_dict.keys())
isp_gr.loc[mask, 'Номенклатура.Код'] = isp_gr.loc[mask, 'Наименование полное'].map(ut_dict)

In [116]:
isp_gr.info(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5083 entries, 0 to 5082
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Номенклатура.Код                 5083 non-null   object        
 1   Договор.Дата окончания действия  5083 non-null   datetime64[ns]
 2   Год поставки                     5083 non-null   object        
 3   МНН                              5083 non-null   object        
 4   Наименование полное              5083 non-null   object        
 5   Комментарий расторжение          5083 non-null   object        
 6   В заявке, уп.                    5083 non-null   object        
 7   В заявке, руб.                   5083 non-null   object        
 8   По графику, уп.                  5083 non-null   object        
 9   По графику, руб.                 5083 non-null   object        
 10  Потребность, уп.                 5083 non-null   float64    

In [117]:
result_ostatki['row_num'] = result_ostatki.groupby('Номенклатура.Код').cumcount()
isp_gr['row_num'] = isp_gr.groupby('Номенклатура.Код').cumcount()

# FULL OUTER JOIN чтобы сохранить все строки из обеих таблиц
merged_df = result_ostatki.merge(
    isp_gr,
    on=['Номенклатура.Код', 'row_num'],
    how='outer'
).drop('row_num', axis=1).sort_values('Номенклатура.Код')

In [118]:
# Объединяем одинаковые столбцы
merged_df['Производитель'] = merged_df['Производитель_x'].fillna(merged_df['Производитель_y'])
merged_df['Наименование полное'] = merged_df['Наименование полное_x'].fillna(merged_df['Наименование полное_y'])
merged_df['МНН'] = merged_df['МНН_x'].fillna(merged_df['МНН_y'])

# Удаляем старые столбцы
merged_df = merged_df.drop(['Производитель_x', 'Производитель_y', 
                           'Наименование полное_x', 'Наименование полное_y',
                           'МНН_x', 'МНН_y'], axis=1)

In [119]:
merged_df['Производитель'] = merged_df['Производитель'].replace(company_mapping)

In [120]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7758 entries, 0 to 7757
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Номенклатура.Код                 7758 non-null   object        
 1   Остаток СГ                       5753 non-null   float64       
 2   ОСГ, %                           5753 non-null   float64       
 3   Годен до                         5753 non-null   datetime64[ns]
 4   Цена за единицу                  5753 non-null   float64       
 5   Остатки, уп.                     5753 non-null   float64       
 6   Остатки, руб.                    5753 non-null   float64       
 7   В резерве, уп.                   5753 non-null   float64       
 8   В резерве, руб.                  5753 non-null   float64       
 9   Цена                             5753 non-null   float64       
 10  Договор.Дата окончания действия  5083 non-null   datetime64[ns]
 

In [121]:
# first_rows = result_ostatki.drop_duplicates('Номенклатура.Код', keep='first')

# # Объединяем только первые строки с второй таблицей
# first_rows_merged = pd.merge(
#     first_rows,
#     isp_gr,
#     on='Номенклатура.Код',
#     how='left',
#     suffixes=('', '_from_df2')
# )

# # Создаем маппинг код -> данные из второй таблицы (только для первых строк)
# mapping_dict = first_rows_merged.set_index('Номенклатура.Код')[isp_gr.columns.difference(['Номенклатура.Код'])].to_dict('index')

# # Применяем маппинг ко всей первой таблице
# result_df = result_ostatki.copy()
# for code, values in mapping_dict.items():
#     mask = result_df['Номенклатура.Код'] == code
#     first_idx = result_df[mask].index[0]  # индекс первой строки с этим кодом
    
#     for col, value in values.items():
#         # Заполняем только первую строку, остальные оставляем как есть (или NaN)
#         result_df.loc[first_idx, col] = value

In [122]:
# # Предполагаем, что у вас уже есть датафреймы isp_gr и result_df
# # Извлекаем уникальные значения из столбца "Номенклатура.Код"
# unique_isp_gr = set(isp_gr['Номенклатура.Код'].unique())
# unique_result_df = set(merged_df['Номенклатура.Код'].unique())

# # Находим значения, которые есть в isp_gr, но отсутствуют в result_df
# missing_values = unique_result_df - unique_isp_gr

In [124]:
# # Добавляем данные потребности, где нет остатков на складе
# result_df = pd.concat([
#     result_df,
#     isp_gr[isp_gr['Номенклатура.Код'].isin(missing_values)]
# ], ignore_index=True)

In [125]:
result_df = merged_df.copy()

In [126]:
result_df['Комментарий расторжение'] = result_df['Комментарий расторжение'].fillna('~')
result_df = result_df.fillna(0)
result_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7758 entries, 0 to 7757
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Номенклатура.Код                 7758 non-null   object 
 1   Остаток СГ                       7758 non-null   float64
 2   ОСГ, %                           7758 non-null   float64
 3   Годен до                         7758 non-null   object 
 4   Цена за единицу                  7758 non-null   float64
 5   Остатки, уп.                     7758 non-null   float64
 6   Остатки, руб.                    7758 non-null   float64
 7   В резерве, уп.                   7758 non-null   float64
 8   В резерве, руб.                  7758 non-null   float64
 9   Цена                             7758 non-null   float64
 10  Договор.Дата окончания действия  7758 non-null   object 
 11  Год поставки                     7758 non-null   object 
 12  Комментарий расторжение  

C:\Users\user078\AppData\Local\Temp\ipykernel_20784\992996106.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result_df = result_df.fillna(0)


In [ ]:
result_df.head()

In [128]:
result_df[result_df['Номенклатура.Код']=='УТ-00000837']['В заявке, уп.'].sum()

np.float64(1829.0)

#### Объединение с дефектурой

In [129]:
result_df_defect = pd.merge(result_df, df_defect, left_on='Номенклатура.Код', right_on='Код', how='left')

In [130]:
result_df_defect.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7758 entries, 0 to 7757
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Номенклатура.Код                 7758 non-null   object        
 1   Остаток СГ                       7758 non-null   float64       
 2   ОСГ, %                           7758 non-null   float64       
 3   Годен до                         7758 non-null   object        
 4   Цена за единицу                  7758 non-null   float64       
 5   Остатки, уп.                     7758 non-null   float64       
 6   Остатки, руб.                    7758 non-null   float64       
 7   В резерве, уп.                   7758 non-null   float64       
 8   В резерве, руб.                  7758 non-null   float64       
 9   Цена                             7758 non-null   float64       
 10  Договор.Дата окончания действия  7758 non-null   object     

In [131]:
#result_df_defect = result_df_defect.drop(['Код', 'key'], axis=1)

In [132]:
result_df_defect['Вид дефектуры'] = result_df_defect['Вид дефектуры'].fillna('Нет дефектуры')
result_df_defect['Дата снятия отметки о дефектуре'] = result_df_defect['Дата снятия отметки о дефектуре'].fillna('~')

In [133]:
result_df_defect.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7758 entries, 0 to 7757
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Номенклатура.Код                 7758 non-null   object 
 1   Остаток СГ                       7758 non-null   float64
 2   ОСГ, %                           7758 non-null   float64
 3   Годен до                         7758 non-null   object 
 4   Цена за единицу                  7758 non-null   float64
 5   Остатки, уп.                     7758 non-null   float64
 6   Остатки, руб.                    7758 non-null   float64
 7   В резерве, уп.                   7758 non-null   float64
 8   В резерве, руб.                  7758 non-null   float64
 9   Цена                             7758 non-null   float64
 10  Договор.Дата окончания действия  7758 non-null   object 
 11  Год поставки                     7758 non-null   object 
 12  Комментарий расторже

In [134]:
result_df_defect = result_df_defect.reindex(columns=[
    'Производитель', 'Номенклатура.Код', 'Наименование полное', 'МНН', 'Договор.Дата окончания действия',
    'Остатки, уп.', 'Остатки, руб.', 'В резерве, уп.', 'В резерве, руб.',
    'В заявке, уп.', 'В заявке, руб.', 'По графику, уп.', 'По графику, руб.', 'Потребность уп./мес.', 'Потребность руб./мес.',
    'Потребность, уп.', 'Потребность, руб.', 'Остаток СГ', 'ОСГ, %', 'Цена за единицу',
    'Вид дефектуры', 'Дата снятия отметки о дефектуре', 'Комментарий', 'Комментарий расторжение', 'Год поставки'
])

In [135]:
result_df_defect = result_df_defect.rename(columns={'Договор.Дата окончания действия':'Договор до'})

In [ ]:
result_df_defect.head(2)

In [151]:
result_df_defect['Вид дефектуры'].unique()

array(['Нет дефектуры', 'Длительная дефектура', 'Снят с производства',
       'Временная дефектура', 'Есть на складе', 'Информирование'],
      dtype=object)

In [152]:
result_df_defect['Комментарий'] = result_df_defect['Комментарий'].fillna('~')

In [153]:
result_df_defect.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7758 entries, 0 to 7757
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Производитель                    7758 non-null   object 
 1   Номенклатура.Код                 7758 non-null   object 
 2   Наименование полное              7758 non-null   object 
 3   МНН                              7758 non-null   object 
 4   Договор до                       7758 non-null   object 
 5   Остатки, уп.                     7758 non-null   float64
 6   Остатки, руб.                    7758 non-null   float64
 7   В резерве, уп.                   7758 non-null   float64
 8   В резерве, руб.                  7758 non-null   float64
 9   В заявке, уп.                    7758 non-null   float64
 10  В заявке, руб.                   7758 non-null   float64
 11  По графику, уп.                  7758 non-null   float64
 12  По графику, руб.    

In [ ]:
result_df_defect.head(2)

#### Добавляем оборачиваемость

In [155]:
#отчет для месячной оборачиваемости: e1cib/data/Справочник.ВариантыОтчетов?ref=865b00505694e4a611f08a333b828d50
#брать данные отдельно за каждый месяц

In [ ]:
import pandas as pd
import os
import glob

# Более гибкий поиск файлов
def find_monthly_files():
    """Находит все Excel файлы в текущей папке"""
    print("Поиск файлов в папке:", os.getcwd())
    
    # Ищем ВСЕ Excel файлы
    all_excel_files = glob.glob('//fs/SCAN/user078/Оборачиваемость по месяцам/*.xlsx') + glob.glob('//fs/SCAN/user078/Оборачиваемость по месяцам/*.xls')
    
    print(f"Найдено Excel файлов: {len(all_excel_files)}")
    
    if all_excel_files:
        print("Список найденных файлов:")
        for i, file in enumerate(all_excel_files, 1):
            print(f"  {i}. {file}")
    
    return all_excel_files

# Загружаем и объединяем данные
def load_and_combine_data(files):
    """Загружает и объединяет данные из всех файлов"""
    all_data = []
    
    for file in files:
        try:
            print(f"Пробуем загрузить: {file}")
            df_month = pd.read_excel(file)
            
            # Проверяем наличие нужных столбцов
            required_columns = ['Номенклатура.Код', 'Номенклатура.Производитель', 
                              'Начальный остаток. Кол-во', 'Расход. Кол-во', 
                              'Конечный остаток. Кол-во']
            
            missing_columns = [col for col in required_columns if col not in df_month.columns]
            if missing_columns:
                print(f"  ⚠ В файле {file} отсутствуют столбцы: {missing_columns}")
                continue
            
            # Добавляем информацию о файле
            df_month['Источник_файл'] = file
            all_data.append(df_month)
            print(f"  ✅ Успешно загружен: {file} ({len(df_month)} записей)")
            
        except Exception as e:
            print(f"  ❌ Ошибка загрузки {file}: {e}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n✅ Всего объединено: {len(combined_df)} записей из {len(all_data)} файлов")
        return combined_df
    else:
        print("❌ Не удалось загрузить ни один файл")
        return pd.DataFrame()

# Рассчитываем оборачиваемость (остается без изменений)
def calculate_turnover(df):
    """Рассчитывает оборачиваемость для каждого препарата"""
    results = []
    
    # Для файлов без месяца используем среднее значение дней
    avg_days_in_month = 30
    
    for code in df['Номенклатура.Код'].unique():
        product_data = df[df['Номенклатура.Код'] == code].copy()
        
        monthly_turnover_days = []
        
        for _, row in product_data.iterrows():
            consumption = row['Расход. Кол-во']
            
            if consumption > 0:
                start_stock = row['Начальный остаток. Кол-во']
                end_stock = row['Конечный остаток. Кол-во']
                
                avg_stock = (start_stock + end_stock) / 2
                
                if avg_stock == 0:
                    turnover_days = 1  # Минимальная оборачиваемость
                else:
                    turnover_days = (avg_stock * avg_days_in_month) / consumption
                
                monthly_turnover_days.append(turnover_days)
        
        # Общая оборачиваемость (средняя по месяцам)
        if monthly_turnover_days:
            overall_turnover = sum(monthly_turnover_days) / len(monthly_turnover_days)
        else:
            overall_turnover = float('inf')
        
        # Рекомендация
        if overall_turnover == float('inf'):
            recommendation = 'Нет движения'
        elif overall_turnover <= 30:
            recommendation = 'Высокая оборачиваемость'
        elif overall_turnover <= 90:
            recommendation = 'Средняя оборачиваемость'
        else:
            recommendation = 'Низкая оборачиваемость'
        
        manufacturer = product_data['Номенклатура.Производитель'].iloc[0]
        
        results.append({
            'Номенклатура.Код': code,
            'Производитель': manufacturer,
            'Оборачиваемость общая': overall_turnover,
            'Рекомендация': recommendation
        })
    
    return pd.DataFrame(results)

# ЗАПУСК ПРОГРАММЫ
print("=" * 60)
print("РАСЧЕТ ОБОРАЧИВАЕМОСТИ ПРЕПАРАТОВ")
print("=" * 60)

# Шаг 1: Находим ВСЕ файлы
print("\n1. Поиск всех Excel файлов...")
files = find_monthly_files()

if not files:
    print("❌ Не найдено Excel файлов в текущей папке!")
    print("\nПроверьте:")
    print("1. Файлы находятся в той же папке, что и скрипт")
    print("2. Файлы имеют расширение .xlsx или .xls")
    print("3. Текущая папка:", os.getcwd())
else:
    # Шаг 2: Загружаем данные
    print(f"\n2. Загрузка {len(files)} файлов...")
    combined_data = load_and_combine_data(files)
    
    if not combined_data.empty:
        # Шаг 3: Расчет оборачиваемости
        unique_products = combined_data['Номенклатура.Код'].nunique()
        print(f"\n3. Расчет оборачиваемости для {unique_products} препаратов...")
        
        df_final_turnover = calculate_turnover(combined_data)
        
        # Шаг 4: Вывод результатов
        print(f"\n4. РЕЗУЛЬТАТЫ:")
        print("=" * 60)
        
        # Форматированный вывод
        df_display = df_final_turnover.copy()
        df_display['Оборачиваемость общая'] = df_display['Оборачиваемость общая'].apply(
            lambda x: '∞' if x == float('inf') else f'{x:.1f} дней'
        )
        
        print(df_display.to_string(index=False))
        
        # Статистика
        print(f"\nСТАТИСТИКА:")
        print(f"Всего препаратов: {len(df_final_turnover)}")
        recommendation_stats = df_final_turnover['Рекомендация'].value_counts()
        for rec, count in recommendation_stats.items():
            print(f"  {rec}: {count} препаратов")
        
        # Шаг 5: Сохранение
        output_file = 'Оборачиваемость_препаратов_итог.xlsx'
        df_final_turnover.to_excel(output_file, index=False)
        print(f"\n✅ Результаты сохранены в файл: {output_file}")
        
        print(f"\n🎯 Готово! Итоговый датафрейм 'df_final_turnover' создан")
        
    else:
        print("❌ Не удалось загрузить данные из файлов")

In [ ]:
import pandas as pd
import os
import glob
import numpy as np

def load_and_combine_data(files):
    """Загружает и объединяет данные из всех файлов"""
    all_data = []
    
    for file in files:
        try:
            print(f"Пробуем загрузить: {file}")
            df_month = pd.read_excel(file)
            
            # Проверяем наличие нужных столбцов
            required_columns = ['Номенклатура.Код', 'Номенклатура.Производитель', 
                              'Начальный остаток. Кол-во', 'Расход. Кол-во', 
                              'Конечный остаток. Кол-во']
            
            missing_columns = [col for col in required_columns if col not in df_month.columns]
            if missing_columns:
                print(f"  ⚠ В файле {file} отсутствуют столбцы: {missing_columns}")
                continue
            
            # Добавляем информацию о файле
            df_month['Источник_файл'] = file
            
            # Извлекаем месяц и год из имени файла
            filename = file.replace('.xlsx', '').replace('.xls', '')
            df_month['МесяцГод'] = filename
            
            all_data.append(df_month)
            print(f"  ✅ Успешно загружен: {file} ({len(df_month)} записей)")
             
        except Exception as e:
            print(f"  ❌ Ошибка загрузки {file}: {e}")
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n✅ Всего объединено: {len(combined_df)} записей из {len(all_data)} файлов")
        return combined_df
    else:
        print("❌ Не удалось загрузить ни один файл")
        return pd.DataFrame()
    
    print(all_data.head())

def calculate_turnover_correct(df):
    """Правильный расчет оборачиваемости - взвешенное среднее"""
    results = []
    
    # Словарь дней в месяцах
    month_days = {
        '01': 31, '02': 28, '03': 31, '04': 30, '05': 31, '06': 30,
        '07': 31, '08': 31, '09': 30, '10': 31, '11': 30, '12': 31,
        '1': 31, '2': 28, '3': 31, '4': 30, '5': 31, '6': 30,
        '7': 31, '8': 31, '9': 30
    }
    
    for code in df['Номенклатура.Код'].unique():
        product_data = df[df['Номенклатура.Код'] == code].copy()
        
        total_stock_days = 0
        total_consumption = 0
        has_any_movement = False
        
        for _, row in product_data.iterrows():
            consumption = row['Расход. Кол-во']
            start_stock = row['Начальный остаток. Кол-во']
            end_stock = row['Конечный остаток. Кол-во']
            
            # Извлекаем месяц из МесяцГод
            month_year = str(row['МесяцГод'])
            month = month_year.split('.')[0] if '.' in month_year else month_year[:2]
            
            # Получаем количество дней в месяце
            days_in_month = month_days.get(month, 30)  # по умолчанию 30 дней
            
            # Проверяем, есть ли вообще какое-то движение товара
            if consumption > 0 or start_stock > 0 or end_stock > 0:
                has_any_movement = True
            
            if consumption > 0:
                avg_stock = (start_stock + end_stock) / 2
                stock_days = avg_stock * days_in_month
                
                total_stock_days += stock_days
                total_consumption += consumption
        
        # РАСЧЕТ ИТОГОВОЙ ОБОРАЧИВАЕМОСТИ (ПРАВИЛЬНЫЙ МЕТОД)
        if total_consumption > 0:
            overall_turnover = total_stock_days / total_consumption
        elif has_any_movement:
            # Товар есть на складе, но не продавался
            overall_turnover = 'Нет движения'  # Высокая оборачиваемость (больше года)
        else:
            # Вообще нет никакого движения
            overall_turnover = 'Нет движения'
        
        # ОБРАБОТКА РЕКОМЕНДАЦИЙ
        if overall_turnover == 'Нет движения':
            recommendation = 'Нет движения'
        elif overall_turnover <= 30:
            recommendation = 'Высокая оборачиваемость'
        elif overall_turnover <= 90:
            recommendation = 'Средняя оборачиваемость'
        else:
            recommendation = 'Низкая оборачиваемость'
        
        manufacturer = product_data['Номенклатура.Производитель'].iloc[0] if len(product_data) > 0 else 'Неизвестно'
        
        results.append({
            'Номенклатура.Код': code,
            'Производитель': manufacturer,
            'Оборачиваемость общая': overall_turnover,
            'Рекомендация': recommendation
        })
    
    return pd.DataFrame(results)

# ОСНОВНОЙ КОД
print("=" * 60)
print("РАСЧЕТ ОБОРАЧИВАЕМОСТИ ПРЕПАРАТОВ (ПРАВИЛЬНЫЙ МЕТОД)")
print("=" * 60)

# Находим файлы
files = glob.glob('*.xlsx') + glob.glob('*.xls')
print(f"Найдено файлов: {len(files)}")

if files:
    # Загружаем данные
    combined_data = load_and_combine_data(files)
    
    if not combined_data.empty:
        # Расчет оборачиваемости ПРАВИЛЬНЫМ методом
        df_final_turnover = calculate_turnover_correct(combined_data)
        
        # Проверяем на наличие проблем
        inf_count = (df_final_turnover['Оборачиваемость общая'] == float('inf')).sum()
        na_count = df_final_turnover['Оборачиваемость общая'].isna().sum()
        
        print(f"\nПРОВЕРКА ДАННЫХ:")
        print(f"Всего препаратов: {len(df_final_turnover)}")
        print(f"С 'inf' (нет движения): {inf_count}")
        print(f"С пустыми значениями: {na_count}")
        
        # Заменяем оставшиеся пустые значения
        if na_count > 0:
            df_final_turnover['Оборачиваемость общая'] = df_final_turnover['Оборачиваемость общая'].fillna('Нет движения')
            print("Пустые значения заменены на 365 дней")
        
        # Вывод результатов
        print(f"\nРЕЗУЛЬТАТЫ (ПРАВИЛЬНЫЙ РАСЧЕТ):")
        #df_display = format_turnover_results(df_final_turnover)
        print(df_display.to_string(index=False))
        
        # Статистика
        print(f"\nСТАТИСТИКА ПО РЕКОМЕНДАЦИЯМ:")
        stats = df_final_turnover['Рекомендация'].value_counts()
        for rec, count in stats.items():
            print(f"  {rec}: {count}")
        
        # Сохранение
        output_file = 'Оборачиваемость_препаратов_ПРАВИЛЬНЫЙ.xlsx'
        df_final_turnover.to_excel(output_file, index=False)
        print(f"\n✅ Результаты сохранены в: {output_file}")
        
        # Дополнительная проверка
        print(f"\nДИАПАЗОН ОБОРАЧИВАЕМОСТИ:")
        valid_turnover = df_final_turnover[df_final_turnover['Оборачиваемость общая'] != float('inf')]['Оборачиваемость общая']
        # if len(valid_turnover) > 0:
        #     print(f"  Минимум: {valid_turnover.min():.1f} дней")
        #     print(f"  Максимум: {valid_turnover.max():.1f} дней")
        #     print(f"  Медиана: {valid_turnover.median():.1f} дней")
        
        print(f"\n🎯 РАСЧЕТ ЗАВЕРШЕН! Использован правильный метод взвешенного среднего")
        
    else:
        print("❌ Не удалось загрузить данные из файлов")
else:
    print("❌ Не найдено Excel файлов в текущей папке!")
    print("Текущая папка:", os.getcwd())

In [167]:
import pandas as pd
import os
from pathlib import Path

# Функция для расчета оборачиваемости за один месяц
def calculate_turnover_for_month(file_path, month_name):
    # Читаем файл
    df = pd.read_excel(file_path)
    
    # Переименуем колонки для удобства
    df = df.rename(columns={
        'Номенклатура.Код': 'code',
        'Номенклатура.Производитель': 'manufacturer', 
        'Номенклатура': 'product_name',
        'Начальный остаток. Кол-во': 'start_qty',
        'Начальный остаток. Себестоимость': 'start_cost',
        'Расход. Кол-во': 'expense_qty', 
        'Расход. Себестоимость': 'expense_cost',
        'Конечный остаток. Кол-во': 'end_qty',
        'Конечный остаток. Себестоимость': 'end_cost'
    })
    
    # Добавляем месяц
    df['month'] = month_name
    
    # Рассчитываем средний запас
    df['avg_stock'] = (df['start_cost'] + df['end_cost']) / 2
    
    # Рассчитываем оборачиваемость в днях
    days_in_month = 30  # Можно уточнить по конкретному месяцу
    
    # Избегаем деления на ноль
    df['turnover_days'] = df.apply(
        lambda x: (x['avg_stock'] * days_in_month) / x['expense_cost'] 
        if x['expense_cost'] > 0 else float('inf'), 
        axis=1
    )
    
    return df[['code', 'manufacturer', 'product_name', 'month', 
               'start_cost', 'expense_cost', 'end_cost', 'avg_stock', 'turnover_days']]

# Основная функция обработки всех файлов
def process_all_months(data_folder):
    all_data = []
    
    # Генерируем список месяцев с 01.23 по 09.25
    months = []
    for year in range(2023, 2026):
        for month in range(1, 13):
            if year == 2025 and month > 10:
                break
            if year == 2023 and month < 1:
                continue
            months.append(f"{month:02d}.{str(year)[-2:]}")
    
    # Обрабатываем каждый месяц
    for month in months:
        file_name = f"{month}.xlsx"
        file_path = os.path.join(data_folder, file_name)
        
        if os.path.exists(file_path):
            print(f"Обрабатывается {file_name}")
            month_data = calculate_turnover_for_month(file_path, month)
            all_data.append(month_data)
        else:
            print(f"Файл {file_name} не найден")
    
    # Объединяем все данные
    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
        return final_df
    else:
        return pd.DataFrame()

# Запускаем обработку
data_folder = "//fs/SCAN/user078/Оборачиваемость по месяцам/"  # Укажите путь к папке с файлами
result = process_all_months(data_folder)

# Сохраняем результат
result.to_excel("оборачиваемость_по_месяцам.xlsx", index=False)
print("Расчет завершен. Результат сохранен в 'оборачиваемость_по_месяцам.xlsx'")

Обрабатывается 01.23.xlsx
Обрабатывается 02.23.xlsx
Обрабатывается 03.23.xlsx
Обрабатывается 04.23.xlsx
Обрабатывается 05.23.xlsx
Обрабатывается 06.23.xlsx
Обрабатывается 07.23.xlsx
Обрабатывается 08.23.xlsx
Обрабатывается 09.23.xlsx
Обрабатывается 10.23.xlsx
Обрабатывается 11.23.xlsx
Обрабатывается 12.23.xlsx
Обрабатывается 01.24.xlsx
Обрабатывается 02.24.xlsx
Обрабатывается 03.24.xlsx
Обрабатывается 04.24.xlsx
Обрабатывается 05.24.xlsx
Обрабатывается 06.24.xlsx
Обрабатывается 07.24.xlsx
Обрабатывается 08.24.xlsx
Обрабатывается 09.24.xlsx
Обрабатывается 10.24.xlsx
Обрабатывается 11.24.xlsx
Обрабатывается 12.24.xlsx
Обрабатывается 01.25.xlsx
Обрабатывается 02.25.xlsx
Обрабатывается 03.25.xlsx
Обрабатывается 04.25.xlsx
Обрабатывается 05.25.xlsx
Обрабатывается 06.25.xlsx
Обрабатывается 07.25.xlsx
Обрабатывается 08.25.xlsx
Обрабатывается 09.25.xlsx
Обрабатывается 10.25.xlsx
Расчет завершен. Результат сохранен в 'оборачиваемость_по_месяцам.xlsx'


In [168]:
def calculate_final_turnover(group):
    total_expense = group['expense_cost'].sum()
    if total_expense == 0:
        return "Нет движения"
    
    # Суммируем средние запасы по месяцам
    total_avg_stock = group['avg_stock'].sum()
    
    # Если сумма средних запасов = 0, но есть расход - используем альтернативный расчет
    if total_avg_stock == 0 and total_expense > 0:
        total_avg_stock = total_expense / 2
    
    turnover_days = (total_avg_stock / len(group) * 1035) / total_expense
    return round(turnover_days, 1)

df_final_turnover = result.groupby('code').apply(calculate_final_turnover).reset_index()
df_final_turnover.columns = ['Номенклатура.Код', 'Оборачиваемость, дн.']
df_final_turnover.to_excel("итоговая_оборачиваемость_по_позициям.xlsx", index=False)

C:\Users\user078\AppData\Local\Temp\ipykernel_20784\2694697420.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_final_turnover = result.groupby('code').apply(calculate_final_turnover).reset_index()


In [169]:
df_final_turnover = pd.read_excel('итоговая_оборачиваемость_по_позициям.xlsx')

In [ ]:
df_final_turnover.head()

In [171]:
result_df_defect_end.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7758 entries, 0 to 7757
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Производитель                    7758 non-null   object 
 1   Номенклатура.Код                 7758 non-null   object 
 2   Наименование полное              7758 non-null   object 
 3   МНН                              7758 non-null   object 
 4   Договор до                       7758 non-null   object 
 5   Остатки, уп.                     7758 non-null   float64
 6   Остатки, руб.                    7758 non-null   float64
 7   В резерве, уп.                   7758 non-null   float64
 8   В резерве, руб.                  7758 non-null   float64
 9   В заявке, уп.                    7758 non-null   float64
 10  В заявке, руб.                   7758 non-null   float64
 11  По графику, уп.                  7758 non-null   float64
 12  По графику, руб.    

In [172]:
df_final_turnover.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4579 entries, 0 to 4578
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Номенклатура.Код      4579 non-null   object
 1   Оборачиваемость, дн.  4579 non-null   object
dtypes: object(2)
memory usage: 71.7+ KB


In [173]:
df_final = pd.merge(result_df_defect_end, df_final_turnover , on=['Номенклатура.Код'], how='left').reset_index(drop=True)

In [174]:
# df_final = df_final.drop(columns='Производитель_y', axis=1)
# df_final = df_final.rename(columns={'Производитель_x':'Производитель'})

In [175]:
df_final = df_final.fillna('Нет движения')

In [ ]:
df_final.head()

In [177]:
# Добавляем категорию
def get_category(turnover):
    if turnover == "Нет движения":
        return "Нет движения"
    try:
        days = float(turnover)
        if days < 30:
            return "Высокая оборачиваемость"
        elif days <= 90:
            return "Средняя оборачиваемость"
        else:
            return "Низкая оборачиваемость"
    except:
        return "Нет движения"

df_final['Категория'] = df_final['Оборачиваемость, дн.'].apply(get_category)
df_final.to_excel("итоговая_оборачиваемость_по_позициям.xlsx", index=False)

In [ ]:
df_final.head()

In [179]:
# Добавляем столбец с остатком срока годности (предполагаем, что он уже есть в данных)
# Обрабатываем строковые значения
def parse_shelf_life(value):
    if pd.isna(value):
        return None
    if value == "Истек":
        return 0
    if value == "Меньше месяца":
        return 0.5  # полмесяца ≈ 15 дней
    try:
        return float(value)  # значение в месяцах
    except:
        return None

df = df_final.copy()

def calculate_turnover_ratio(row):
    if row['Оборачиваемость, дн.'] == "Нет движения":
        return "Нет движения"
    
    shelf_life_months = parse_shelf_life(row['Остаток СГ'])
    if shelf_life_months is None or shelf_life_months == 0:
        return "Н/Д"
    
    # Переводим месяцы в дни (30 дней в месяце)
    shelf_life_days = max(shelf_life_months * 30 - 120, 1)
    turnover_days = float(row['Оборачиваемость, дн.'])
    ratio = turnover_days / shelf_life_days
    return round(ratio, 2)

df_final['Коэффициент_Оборачиваемость_СГ'] = df_final.apply(calculate_turnover_ratio, axis=1)
df_final.to_excel("итоговая_оборачиваемость_с_коэффициентом.xlsx", index=False)

#### Вывод

In [180]:
df_final['Договор до'] = pd.to_datetime(df_final['Договор до'], errors='coerce')

In [181]:
df_final['Остатки, руб.'].sum()

np.float64(771985854.5341854)

In [182]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7758 entries, 0 to 7757
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Производитель                    7758 non-null   object        
 1   Номенклатура.Код                 7758 non-null   object        
 2   Наименование полное              7758 non-null   object        
 3   МНН                              7758 non-null   object        
 4   Договор до                       7758 non-null   datetime64[ns]
 5   Остатки, уп.                     7758 non-null   float64       
 6   Остатки, руб.                    7758 non-null   float64       
 7   В резерве, уп.                   7758 non-null   float64       
 8   В резерве, руб.                  7758 non-null   float64       
 9   В заявке, уп.                    7758 non-null   float64       
 10  В заявке, руб.                   7758 non-null   float64    

In [183]:
# Добавляем столбец с товарной категорией
tov_cat = pd.read_excel('//fs/SCAN/user078/Товарная категория.xlsx')
tov_cat = tov_cat.rename(columns={'Код':'Номенклатура.Код'})
tov_cat = tov_cat.drop_duplicates()
tov_cat = tov_cat[~tov_cat['Номенклатура.Код'].duplicated()]

In [184]:
df_final = pd.merge(df_final, tov_cat[['Номенклатура.Код', 'Товарная категория']],
                    on='Номенклатура.Код', how='left').reset_index(drop=True)

In [ ]:
df_result_ostatki = df_final.groupby(['Производитель', 'Номенклатура.Код', 'Наименование полное',
                                     'Остаток СГ']).agg({'Остатки, уп.':'sum', 'Остатки, руб.':'sum'}).reset_index()
df_result_ostatki.head()

In [ ]:
df_result_potreb = df_final.groupby(['Производитель', 'Номенклатура.Код',
                                     'Наименование полное', 'Договор до']).agg({
                                                                 'Потребность, уп.':'sum',
                                                                 'Потребность, руб.':'sum'}).reset_index()
df_result_potreb.head()

In [189]:
df_result_potreb_2025 = df_result_potreb[(df_result_potreb['Договор до']!='1970.01.01') & (df_result_potreb['Договор до']<'2026.01.01')]

In [190]:
df_result_potreb.to_excel('Потребность 2025.xlsx', index=False)

In [191]:
df_result_potreb_2026 = df_result_potreb[(df_result_potreb['Договор до']>'2025.12.31') & (df_result_potreb['Договор до']<'2027.01.01')]

In [192]:
df_result_potreb_2025['Потребность, руб.'].sum()

np.float64(445817150.1)

In [193]:
df_result_potreb.to_excel('Потребность 2026.xlsx', index=False)

In [194]:
#df_final.to_excel('test.xlsx', index=False)

In [195]:
# Сохранение сводной таблицы с остатками в excel файл
today = datetime.now().strftime("%Y-%m-%d")
#filename = f"Для_дашборда_30.09.2025.xlsx"
filename = f"Данные с оборачиваемостью {today}.xlsx"
# Сохраняем С индексом (чтобы не потерять Производителя)
with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
    df_final.to_excel(writer, sheet_name='Остатки', index=True)  # index=True!
    
    workbook = writer.book
    worksheet = writer.sheets['Остатки']
    
    # Форматирование
    format_header = workbook.add_format({
        'bold': True,
        'border': 1,
        'bg_color': '#FFE699',
        'align': 'center'
    })
    
    # Применяем к заголовкам (учитываем, что теперь есть индекс)
    for col_num, value in enumerate(df_final.index.names + list(df_final.columns.values)):
        worksheet.write(0, col_num, value, format_header)
    
    # Авто-ширина колонок
    worksheet.set_column(0, len(df_final.index.names) + len(df_final.columns) - 1, 20)

In [196]:
calculator_df = df_final[['Номенклатура.Код', 'Наименование полное', 'Договор до', 'Остатки, уп.', 'Остатки, руб.',
                         'Потребность, уп.', 'Потребность, руб.', 'Потребность уп./мес.', 'Потребность руб./мес.']]

calculator_df.head()

,Номенклатура.Код,Наименование полное,Договор до,"Остатки, уп.","Остатки, руб.","Потребность, уп.","Потребность, руб.",Потребность уп./мес.,Потребность руб./мес.
0,,"Панкреатин форте, табл. кишечнораствор. п.п.о....",2025-12-31,0.0,0.0,2116.998,1070784.08,2116.0,1070787.0
1,,"Рокуроний, р-р для в/в введ. (sol. ad introduc...",2027-02-28,0.0,0.0,1652.000,353048.28,115.0,23540.0
2,,"Рокуроний, р-р для в/в введ. (sol. ad introduc...",2027-02-28,0.0,0.0,9.000,5143.05,1.0,343.0
3,,"Кеторолак Велфарм, р-р для в/в и в/м введ. (so...",2027-01-22,0.0,0.0,658.000,36584.80,47.0,2613.0
4,,"Симбикорт Турбухалер, пор. д/ингал. доз. (pulv...",2026-12-31,0.0,0.0,21175.999,9194385.64,1632.0,707259.0


In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')

class InventoryCoverageCalculator:
    def __init__(self):
        self.stock_data = None
        self.contract_data = None
        self.monthly_demand = None
        self.coverage_results = None
    
    def prepare_data(self, raw_data):
        """Подготовка и разделение исходных данных"""
        # Создаем DataFrame из сырых данных
        df = pd.DataFrame(raw_data)
        
        # Разделяем на таблицы остатков и контрактов
        self._create_stock_table(df)
        self._create_contract_table(df)
        
        print("=== ПОДГОТОВЛЕННЫЕ ДАННЫЕ ===")
        print("\n1. ОСТАТКИ НА СКЛАДЕ:")
        print(self.stock_data)
        print("\n2. ДАННЫЕ ПО КОНТРАКТАМ:")
        print(self.contract_data)
    
    def _create_stock_table(self, df):
        """Создаем таблицу остатков"""
        stock_data = []
        for sku in df['Номенклатура.Код'].unique():
            sku_data = df[df['Номенклатура.Код'] == sku]
            # Берем первый ненулевой остаток для каждого SKU
            stock_row = sku_data[sku_data['Остатки, уп.'] > 0]
            if not stock_row.empty:
                stock = stock_row.iloc[0]['Остатки, уп.']
            else:
                stock = 0
            stock_data.append({'Номенклатура.Код': sku, 'Остатки_уп': stock})
        
        self.stock_data = pd.DataFrame(stock_data)
    
    def _create_contract_table(self, df):
        """Создаем таблицу контрактов"""
        contract_data = []
        for _, row in df.iterrows():
            if row['Потребность, уп.'] > 0:  # Только строки с потребностью
                contract_data.append({
                    'Номенклатура.Код': row['Номенклатура.Код'],
                    'Договор_до': pd.to_datetime(row['Договор до']),
                    'Общая_потребность_уп': row['Потребность, уп.'],
                    'Месячная_потребность_уп': row['Потребность уп./мес.']
                })
        
        self.contract_data = pd.DataFrame(contract_data)
    
    def create_monthly_demand_plan(self, start_date=None):
        """Создаем помесячный план отгрузок"""
        if start_date is None:
            start_date = datetime.now().replace(day=1)  # Начало текущего месяца
        
        monthly_plan = []
        
        for _, contract in self.contract_data.iterrows():
            sku = contract['Номенклатура.Код']
            end_date = contract['Договор_до']
            total_demand = contract['Общая_потребность_уп']
            monthly_rate = contract['Месячная_потребность_уп']
            
            # Определяем дату начала (текущий месяц или дата контракта, если в будущем)
            contract_start = start_date
            
            current_date = contract_start
            remaining_demand = total_demand
            
            # Распределяем потребность по месяцам
            while remaining_demand > 0 and current_date <= end_date:
                # Определяем потребность на текущий месяц
                month_demand = min(monthly_rate, remaining_demand)
                
                monthly_plan.append({
                    'Номенклатура.Код': sku,
                    'Месяц': current_date.strftime('%Y-%m'),
                    'Год': current_date.year,
                    'Месяц_число': current_date.month,
                    'Потребность_уп': month_demand
                })
                
                remaining_demand -= month_demand
                current_date += relativedelta(months=1)
        
        self.monthly_demand = pd.DataFrame(monthly_plan)
        
        print("\n3. ПОМЕСЯЧНЫЙ ПЛАН ОТГРУЗОК:")
        print(self.monthly_demand.to_string(index=False))
    
    def calculate_coverage(self):
        """Рассчитываем покрытие запасов по методу 2"""
        results = []
        
        for sku in self.stock_data['Номенклатура.Код'].unique():
            # Получаем остаток для SKU
            stock = self.stock_data[
                self.stock_data['Номенклатура.Код'] == sku
            ]['Остатки_уп'].iloc[0]
            
            # Получаем помесячный спрос для SKU, отсортированный по дате
            sku_demand = self.monthly_demand[
                self.monthly_demand['Номенклатура.Код'] == sku
            ].sort_values(['Год', 'Месяц_число'])
            
            print(f"\n--- РАСЧЕТ ДЛЯ {sku} (остаток: {stock} уп.) ---")
            
            if sku_demand.empty:
                # Если нет спроса, покрытие бесконечное
                results.append({
                    'Номенклатура.Код': sku,
                    'Остаток_уп': stock,
                    'Покрытие_мес': float('inf'),
                    'Месяц_исчерпания': 'Нет спроса',
                    'Остаток_последний': stock
                })
                continue
            
            remaining_stock = stock
            full_months = 0
            partial_coverage = 0
            depletion_month = None
            
            for _, month_data in sku_demand.iterrows():
                month = month_data['Месяц']
                demand = month_data['Потребность_уп']
                
                if remaining_stock <= 0:
                    break
                
                if remaining_stock >= demand:
                    # Хватает на полный месяц
                    remaining_stock -= demand
                    full_months += 1
                    print(f"  {month}: {demand} уп. → остаток {remaining_stock} уп. (полный месяц)")
                else:
                    # Хватает только на часть месяца
                    partial_coverage = remaining_stock / demand
                    depletion_month = month
                    print(f"  {month}: {demand} уп. → остаток {remaining_stock} уп. → покрытие {partial_coverage:.2f} месяца")
                    remaining_stock = -1  # Помечаем как исчерпанный
                    break
            
            if remaining_stock > 0:
                # Запас не исчерпан
                coverage = full_months
                depletion_month = "Не исчерпан"
            else:
                coverage = full_months + partial_coverage
                if depletion_month is None:
                    depletion_month = "Текущий месяц"
            
            results.append({
                'Номенклатура.Код': sku,
                'Остаток_уп': stock,
                'Покрытие_мес': coverage,
                'Месяц_исчерпания': depletion_month,
                'Остаток_последний': max(0, remaining_stock)
            })
        
        self.coverage_results = pd.DataFrame(results)
        return self.coverage_results
    
    def generate_dashboard_data(self):
        """Генерируем данные для дашборда"""
        if self.coverage_results is None:
            self.calculate_coverage()
        
        print("\n" + "="*60)
        print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ РАСЧЕТА ПОКРЫТИЯ")
        print("="*60)
        
        # Сводные метрики
        critical_items = len(self.coverage_results[
            self.coverage_results['Покрытие_мес'] < 1
        ])
        excess_items = len(self.coverage_results[
            self.coverage_results['Покрытие_мес'] > 6
        ])
        
        print(f"\nСВОДНЫЕ ПОКАЗАТЕЛИ:")
        print(f"• Товаров с покрытием < 1 месяца: {critical_items}")
        print(f"• Товаров с покрытием > 6 месяцев: {excess_items}")
        print(f"• Среднее покрытие: {self.coverage_results['Покрытие_мес'].mean():.2f} мес.")
        
        print(f"\nДЕТАЛИЗИРОВАННЫЕ РЕЗУЛЬТАТЫ:")
        for _, result in self.coverage_results.iterrows():
            status = "🔴 КРИТИЧЕСКИЙ" if result['Покрытие_мес'] < 1 else \
                    "🟡 ВНИМАНИЕ" if result['Покрытие_мес'] < 3 else "🟢 НОРМА"
            
            print(f"{result['Номенклатура.Код']}: {result['Покрытие_мес']:.2f} мес. "
                  f"(остаток: {result['Остаток_уп']} уп.) → {status}")
        
        return self.coverage_results

# ИСПОЛЬЗОВАНИЕ
def main():
    # Ваши исходные данные
    # raw_data = {
    #     'Номенклатура.Код': ['УТ-00000344', 'УТ-00000344', 'УТ-00000371', 'УТ-00000578'],
    #     'Договор до': ['31.01.2026', '20.12.2025', '30.12.2025', '06.05.2026'],
    #     'Остатки, уп.': [0, 65, 72, 0],
    #     'Остатки, руб.': [0, 7529.31, 2880, 0],
    #     'Потребность уп./мес.': [25.00, 3.00, 4.00, 65.00],
    #     'Потребность руб./мес.': [2815.00, 234.00, 81.00, 1790.00],
    #     'Потребность уп.': [76.00, 7.00, 8.00, 390.00],
    #     'Потребность руб.': [8444.29, 468.00, 162.91, 10742.73]
    # }

    raw_data = calculator_df.copy()
    
    # Создаем калькулятор и выполняем расчет
    calculator = InventoryCoverageCalculator()
    calculator.prepare_data(calculator_df)
    calculator.create_monthly_demand_plan()
    results = calculator.calculate_coverage()
    dashboard = calculator.generate_dashboard_data()
    
    return calculator, results

if __name__ == "__main__":
    calculator, results = main()

In [198]:
# КОМПАКТНАЯ ВЕРСИЯ ДЛЯ БЫСТРОГО ИСПОЛЬЗОВАНИЯ:
def quick_coverage_calculation(calculator_df, start_date=None):
    """Компактная версия без лишних выводов"""
    if start_date is None:
        start_date = datetime.now().replace(day=1)
    
    # 1. Агрегируем остатки
    stock_data = calculator_df.groupby('Номенклатура.Код', as_index=False)['Остатки, уп.'].sum()
    
    # 2. Создаем помесячный план
    calculator_df['Договор_до'] = pd.to_datetime(calculator_df['Договор до'], dayfirst=True)
    monthly_plan = []
    
    for sku in calculator_df['Номенклатура.Код'].unique():
        sku_data = calculator_df[calculator_df['Номенклатура.Код'] == sku]
        monthly_demand = {}
        
        for _, row in sku_data.iterrows():
            if pd.isna(row['Договор_до']) or row['Потребность, уп.'] <= 0:
                continue
                
            current_date = start_date
            remaining = row['Потребность, уп.']
            monthly_rate = row['Потребность уп./мес.']
            
            while remaining > 0 and current_date <= row['Договор_до']:
                month_key = current_date.strftime('%Y-%m')
                demand = min(monthly_rate, remaining)
                
                monthly_demand[month_key] = monthly_demand.get(month_key, 0) + demand
                remaining -= demand
                current_date += relativedelta(months=1)
        
        for month, demand in monthly_demand.items():
            monthly_plan.append({'Номенклатура.Код': sku, 'Месяц': month, 'Потребность_уп': demand})
    
    monthly_demand_df = pd.DataFrame(monthly_plan)
    
    # 3. Рассчитываем покрытие
    results = []
    for _, stock_row in stock_data.iterrows():
        sku, stock = stock_row['Номенклатура.Код'], stock_row['Остатки, уп.']
        
        sku_months = monthly_demand_df[monthly_demand_df['Номенклатура.Код'] == sku].sort_values('Месяц')
        
        if sku_months.empty:
            results.append({'Номенклатура.Код': sku, 'Остаток_уп': stock, 'Покрытие_мес': 'Нет спроса'})
            continue
        
        remaining = stock
        coverage = 0
        
        for _, month_row in sku_months.iterrows():
            if remaining <= 0:
                break
            if remaining >= month_row['Потребность_уп']:
                remaining -= month_row['Потребность_уп']
                coverage += 1
            else:
                coverage += remaining / month_row['Потребность_уп']
                remaining = -1
                break
        
        results.append({
            'Номенклатура.Код': sku, 
            'Остаток_уп': stock,
            'Покрытие_мес': round(coverage, 2)
        })
    
    return pd.DataFrame(results)

results = quick_coverage_calculation(calculator_df)

In [199]:
results.info(30)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2272 entries, 0 to 2271
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Номенклатура.Код  2272 non-null   object 
 1   Остаток_уп        2272 non-null   float64
 2   Покрытие_мес      2272 non-null   object 
dtypes: float64(1), object(2)
memory usage: 53.4+ KB


In [ ]:
results_name = pd.merge(results, df_final[['Номенклатура.Код', 'Наименование полное', 'Остатки, руб.']], on='Номенклатура.Код', how='left')
results_name.head()

In [201]:
results_name_gr = results_name.groupby(['Номенклатура.Код', 'Наименование полное', 'Остаток_уп', 'Покрытие_мес']).agg({'Остатки, руб.':'sum'}).reset_index()

In [202]:
results_name_gr['Остатки, руб.'].sum()

np.float64(771985854.5341854)

### Вывод остатков и потребности отдельно за 2025 и 2026 годы для расчет дельты (остатки - потребность)

In [ ]:
df_final.head(3)

In [8]:
# Пример данных

demo = pd.DataFrame([
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Ном_010",
        "Наименование_анон": "Препарат_001",
        "МНН_анон": "МНН_D",
        "Договор до": "2025-12-31",
        "Остатки_уп_анон": 0.0,
        "Остатки_руб_анон": 0.0,
        "В резерве_уп_анон": 0.0,
        "В резерве_руб_анон": 0.0,
        "В заявке_уп_анон": 81.9,          # 819.0 / 10
        "В заявке_руб_анон": 7769.63,      # 77696.345455 / 10
        "По графику_уп_анон": 6.0,         # 60.0 / 10
        "По графику_руб_анон": 11195.44,   # 111954.363636 / 10
        "Потр._уп/мес_анон": 211.7,        # 2116.0 / 10
        "Потр._руб/мес_анон": 107078.7,    # 1070787.0 / 10
        "Потребность_уп_анон": 211.7,      # 2116.998 / 10
        "Потребность_руб_анон": 107078.4,  # 1070784.08 / 10
        "Остаток СГО_анон": 0.0,
        "СГОСГ_анон_%": 0.0,
        "Цена_анон": 0.0,
        "Вид дефектуры": "Нет дефектуры",
        "Дата снятия дефектуры": "2025-12-05",
        "Комментарий_анон": "Комментарий_001",
        "Год поставки": 2025,
        "Доля истекшего товара": 0.0,
        "Оборачиваемость_дн_анон": "Нет движения",
        "Категория_анон": "Нет движения",
        "Коэф.Оборач._СГ_анон": np.nan,
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Ном_011",
        "Наименование_анон": "Препарат_001",
        "МНН_анон": "МНН_E",
        "Договор до": "2027-02-28",
        "Остатки_уп_анон": 0.0,
        "Остатки_руб_анон": 0.0,
        "В резерве_уп_анон": 0.0,
        "В резерве_руб_анон": 0.0,
        "В заявке_уп_анон": 11.2,          # 112.0 / 10
        "В заявке_руб_анон": 5733.79,      # 57337.890909 / 10
        "По графику_уп_анон": 0.0,
        "По графику_руб_анон": 0.0,
        "Потр._уп/мес_анон": 165.2,        # 1652.000 / 10
        "Потр._руб/мес_анон": 35304.83,    # 353048.28 / 10
        "Потребность_уп_анон": 165.2,      # 115.0 + 50.2 условно → можно упростить
        "Потребность_руб_анон": 35304.83,
        "Остаток СГО_анон": 0.0,
        "СГОСГ_анон_%": 0.0,
        "Цена_анон": 0.0,
        "Вид дефектуры": "Нет дефектуры",
        "Дата снятия дефектуры": None,
        "Комментарий_анон": "Комментарий_002",
        "Год поставки": 2026,
        "Доля истекшего товара": 0.0,
        "Оборачиваемость_дн_анон": "Нет движения",
        "Категория_анон": "Нет движения",
        "Коэф.Оборач._СГ_анон": np.nan,
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Ном_011",
        "Наименование_анон": "Препарат_001",
        "МНН_анон": "МНН_E",
        "Договор до": "2027-02-28",
        "Остатки_уп_анон": 0.0,
        "Остатки_руб_анон": 0.0,
        "В резерве_уп_анон": 0.0,
        "В резерве_руб_анон": 0.0,
        "В заявке_уп_анон": 0.0,
        "В заявке_руб_анон": 0.0,
        "По графику_уп_анон": 0.1,         # 1.0 / 10
        "По графику_руб_анон": 50.73,      # 507.272727 / 10
        "Потр._уп/мес_анон": 9.0,
        "Потр._руб/мес_анон": 514.31,      # 5143.05 / 10
        "Потребность_уп_анон": 9.0,
        "Потребность_руб_анон": 514.31,
        "Остаток СГО_анон": 0.0,
        "СГОСГ_анон_%": 0.0,
        "Цена_анон": 0.0,
        "Вид дефектуры": "Нет дефектуры",
        "Дата снятия дефектуры": None,
        "Комментарий_анон": "Комментарий_002",
        "Год поставки": 2025,
        "Доля истекшего товара": 0.0,
        "Оборачиваемость_дн_анон": "Нет движения",
        "Категория_анон": "Нет движения",
        "Коэф.Оборач._СГ_анон": np.nan,
    },
])

demo

,Производитель_анон,Код_анон,Наименование_анон,МНН_анон,Договор до,Остатки_уп_анон,Остатки_руб_анон,В резерве_уп_анон,В резерве_руб_анон,В заявке_уп_анон,В заявке_руб_анон,По графику_уп_анон,По графику_руб_анон,Потр._уп/мес_анон,Потр._руб/мес_анон,Потребность_уп_анон,Потребность_руб_анон,Остаток СГО_анон,СГОСГ_анон_%,Цена_анон,Вид дефектуры,Дата снятия дефектуры,Комментарий_анон,Год поставки,Доля истекшего товара,Оборачиваемость_дн_анон,Категория_анон,Коэф.Оборач._СГ_анон
0,Производитель_001,Ном_010,Препарат_001,МНН_D,2025-12-31,0.0,0.0,0.0,0.0,81.9,7769.63,6.0,11195.44,211.7,107078.70,211.7,107078.40,0.0,0.0,0.0,Нет дефектуры,2025-12-05,Комментарий_001,2025,0.0,Нет движения,Нет движения,NaN
1,Производитель_001,Ном_011,Препарат_001,МНН_E,2027-02-28,0.0,0.0,0.0,0.0,11.2,5733.79,0.0,0.00,165.2,35304.83,165.2,35304.83,0.0,0.0,0.0,Нет дефектуры,NaN,Комментарий_002,2026,0.0,Нет движения,Нет движения,NaN
2,Производитель_001,Ном_011,Препарат_001,МНН_E,2027-02-28,0.0,0.0,0.0,0.0,0.0,0.00,0.1,50.73,9.0,514.31,9.0,514.31,0.0,0.0,0.0,Нет дефектуры,NaN,Комментарий_002,2025,0.0,Нет движения,Нет движения,NaN


In [204]:
df_final_delta = df_final.copy()

In [205]:
df_final_delta['Остатки общие, уп.'] = df_final_delta.groupby('Номенклатура.Код')['Остатки, уп.'].transform('sum')
df_final_delta['Остатки общие, руб.'] = df_final_delta.groupby('Номенклатура.Код')['Остатки, руб.'].transform('sum')

In [206]:
df_final_delta['Год'] = df_final_delta['Договор до'].dt.year

In [207]:
df_final_delta = df_final_delta[df_final_delta['Год']!=1970]

In [208]:
df_final_delta['В заявке, уп.'] = df_final_delta['В заявке, уп.'] + df_final_delta['По графику, уп.']
df_final_delta['В заявке, руб.'] = df_final_delta['В заявке, руб.'] + df_final_delta['По графику, руб.']

In [209]:
df_final_delta = df_final_delta.groupby(['Номенклатура.Код', 'Наименование полное',
                                         'Остатки общие, уп.', 'Остатки общие, руб.',
                                         'Вид дефектуры', 'Год', 'Комментарий']).agg({'В заявке, уп.':'sum',
                                                                'В заявке, руб.':'sum',
                                                                'Потребность, уп.':'sum',
                                                                'Потребность, руб.':'sum'}).reset_index()

In [ ]:
df_final_delta.head(3)

In [211]:
df_final_delta['Потребность, уп.'].sum()

np.float64(5872041.102)

In [212]:
df_final_delta.to_excel('Дельта по годам.xlsx', index=False)

#### Дельта остатки - потребность общие

In [213]:
df_final_delta_full = df_final.copy()

In [214]:
df_final_delta_full['Остатки общие, уп.'] = df_final_delta_full.groupby('Номенклатура.Код')['Остатки, уп.'].transform('sum')
df_final_delta_full['Остатки общие, руб.'] = df_final_delta_full.groupby('Номенклатура.Код')['Остатки, руб.'].transform('sum')

In [215]:
df_final_delta_full['В заявке, уп.'] = df_final_delta_full['В заявке, уп.'] + df_final_delta_full['По графику, уп.']
df_final_delta_full['В заявке, руб.'] = df_final_delta_full['В заявке, руб.'] + df_final_delta_full['По графику, руб.']

In [216]:
df_final_delta_full['В заявке, уп.'].sum()

np.float64(311117.81799999997)

In [217]:
df_final_delta_full['Товарная категория'] = df_final_delta_full['Товарная категория'].fillna('~')

In [218]:
df_final_delta_full = df_final_delta_full.groupby(['Номенклатура.Код',
                                         'Остатки общие, уп.', 'Остатки общие, руб.',
                                         'Вид дефектуры', 'Комментарий', 'Товарная категория']).agg({'В заявке, уп.':'sum',
                                                                                                     'В заявке, руб.':'sum',
                                                                                                     'Потребность, уп.':'sum',
                                                                                                     'Потребность, руб.':'sum'}).reset_index()

In [219]:
df_final_delta_full['В заявке, уп.'].sum()

np.float64(311117.81799999997)

In [220]:
mask_name = df_final[['Номенклатура.Код', 'Наименование полное']]

In [221]:
mask_name = mask_name[~mask_name['Номенклатура.Код'].duplicated()]

In [222]:
df_final_delta_full = pd.merge(df_final_delta_full, mask_name, on='Номенклатура.Код', how='left').reset_index()

In [ ]:
df_final_delta_full['Потребность, уп.'].sum()

In [224]:
df_final_delta_full.to_excel('Дельта общее.xlsx', index=False)

In [225]:
df_final_delta_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2272 entries, 0 to 2271
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                2272 non-null   int64  
 1   Номенклатура.Код     2272 non-null   object 
 2   Остатки общие, уп.   2272 non-null   float64
 3   Остатки общие, руб.  2272 non-null   float64
 4   Вид дефектуры        2272 non-null   object 
 5   Комментарий          2272 non-null   object 
 6   Товарная категория   2272 non-null   object 
 7   В заявке, уп.        2272 non-null   float64
 8   В заявке, руб.       2272 non-null   float64
 9   Потребность, уп.     2272 non-null   float64
 10  Потребность, руб.    2272 non-null   float64
 11  Наименование полное  2272 non-null   object 
dtypes: float64(6), int64(1), object(5)
memory usage: 213.1+ KB
